
# 02 — Temporal Window & Channel Feasibility Audit

## 1. Data Loading and Integrity Validation

### Objective
Prepare and validate the user-level and daily Future-period datasets required for temporal window analysis.

### Input Datasets
- `user_population_profile.parquet`: Consolidated behavioral profile containing 101 features for 3,995 benign users.
- `behavior_features_r62_v1.parquet`: Original Data Engineering features for eligible Future user-workdays.

### Validation Checks
- Confirm that all required files and columns exist.
- Verify unique user identifiers and user-day records.
- Confirm that all daily records belong to the Future period.
- Ensure that daily users are included in the population profile.
- Compare each user's daily record count against `future_workdays`.
- Validate Email, USB, and HTTP Upload activity counts.
- Confirm that no users or eligible workdays are lost during loading.

### Methodology
- **User-First:** Available windows and channel patterns will be discovered from each user's actual data.
- **Baseline Protection:** Historical Baseline data will not be modified or recalculated.
- **No Predefined Scenarios:** No attack duration, switching sequence, or eligibility threshold will be imposed at this stage.
- **No Injection:** This notebook performs feasibility analysis only.

### Expected Result
Two validated in-memory datasets:
- `user_profile`
- `future_daily`

No output files are generated in this section.


In [1]:

# ============================================================
# NOTEBOOK 02 — TEMPORAL WINDOW & CHANNEL FEASIBILITY AUDIT
# SECTION 1: DATA LOADING AND INTEGRITY VALIDATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

DATA_ROOT = (
    PROJECT_ROOT.parent
    / "Data_Engineering"
    / "Data_engineering"
    / "data"
)

PROFILE_PATH = (
    PROJECT_ROOT
    / "reports"
    / "01_User_Population_Audit"
    / "user_population_profile.parquet"
)

DAILY_PATH = (
    DATA_ROOT
    / "features"
    / "behavior_features_r62_v1.parquet"
)

REPORT_DIR_02 = (
    PROJECT_ROOT
    / "reports"
    / "02_Temporal_Window_Audit"
)

for path in [PROFILE_PATH, DAILY_PATH]:
    assert path.is_file(), f"Missing input file: {path}"

# ------------------------------------------------------------
# 2. LOAD DATASETS — READ ONLY
# ------------------------------------------------------------

user_profile = pd.read_parquet(PROFILE_PATH)

daily_columns = [
    "user",
    "day",
    "external_email_count",
    "usb_copy_count",
    "www_upload_count"
]

future_daily = pd.read_parquet(
    DAILY_PATH,
    columns=daily_columns
)

future_daily["day"] = pd.to_datetime(
    future_daily["day"]
)

# ------------------------------------------------------------
# 3. BASIC SCHEMA & KEY VALIDATION
# ------------------------------------------------------------

required_profile_columns = [
    "user",
    "future_workdays",
    "has_valid_baseline",
    "has_future_workdays",
    "email_days",
    "usb_days",
    "http_days",
    "observed_channel_set"
]

assert set(required_profile_columns).issubset(
    user_profile.columns
)

assert user_profile["user"].is_unique
assert user_profile["user"].notna().all()

assert not future_daily.duplicated(
    ["user", "day"]
).any()

assert future_daily[
    ["user", "day"]
].notna().all().all()

assert user_profile["has_valid_baseline"].all()

# ------------------------------------------------------------
# 4. FUTURE PERIOD VALIDATION
# ------------------------------------------------------------

FUTURE_START = pd.Timestamp("2011-01-01")
FUTURE_END = pd.Timestamp("2011-06-01")

assert future_daily["day"].between(
    FUTURE_START, FUTURE_END
).all()

# All daily users must exist in the consolidated profile.
profile_users = set(user_profile["user"])
daily_users = set(future_daily["user"])

assert daily_users.issubset(profile_users)

# Users with eligible Future days must match exactly.
expected_future_users = set(
    user_profile.loc[
        user_profile["future_workdays"] > 0,
        "user"
    ]
)

assert daily_users == expected_future_users

# ------------------------------------------------------------
# 5. USER-DAY COVERAGE VALIDATION
# ------------------------------------------------------------

daily_counts = (
    future_daily.groupby("user")
    .size()
)

expected_counts = (
    user_profile["user"]
    .map(daily_counts)
    .fillna(0)
    .astype(int)
)

assert (
    expected_counts.to_numpy()
    == user_profile["future_workdays"].to_numpy()
).all()

assert (
    user_profile["has_future_workdays"].to_numpy()
    == user_profile["future_workdays"].gt(0).to_numpy()
).all()

# ------------------------------------------------------------
# 6. CHANNEL ACTIVITY VALIDATION
# ------------------------------------------------------------

CHANNELS = {
    "email": "external_email_count",
    "usb": "usb_copy_count",
    "http": "www_upload_count"
}

for channel, column in CHANNELS.items():

    values = future_daily[column]

    assert pd.api.types.is_numeric_dtype(values)
    assert values.notna().all()
    assert np.isfinite(values.to_numpy()).all()
    assert (values >= 0).all()

    actual_days = (
        future_daily[column].gt(0)
        .groupby(future_daily["user"])
        .sum()
    )

    expected_days = (
        user_profile["user"]
        .map(actual_days)
        .fillna(0)
        .astype(int)
    )

    assert (
        expected_days.to_numpy()
        == user_profile[f"{channel}_days"].to_numpy()
    ).all(), f"{channel} activity mismatch"

# ------------------------------------------------------------
# 7. PREPARE CHRONOLOGICAL DAILY DATA
# ------------------------------------------------------------

future_daily = (
    future_daily
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

for channel, column in CHANNELS.items():

    future_daily[f"{channel}_active"] = (
        future_daily[column].gt(0).astype("int8")
    )

future_daily["active_channel_count"] = (
    future_daily[
        ["email_active", "usb_active", "http_active"]
    ].sum(axis=1).astype("int8")
)

# ------------------------------------------------------------
# 8. CONSOLIDATED VALIDATION REPORT
# ------------------------------------------------------------

print("=" * 72)
print("NOTEBOOK 02 — DATA LOADING & INTEGRITY")
print("=" * 72)

print(f"Population users       : {len(user_profile):,}")
print(f"Future-active users    : {len(daily_users):,}")
print(f"Users without Future   : {len(profile_users - daily_users):,}")
print(f"Eligible Future days   : {len(future_daily):,}")
print(f"Profile features       : {len(user_profile.columns)}")

print("\nDATA COVERAGE")
print("-" * 72)
print("First observed day     :", future_daily["day"].min().date())
print("Last observed day      :", future_daily["day"].max().date())

print("\nCHANNEL COVERAGE")
print("-" * 72)

for channel in CHANNELS:
    print(
        f"{channel.upper():<8} active users:",
        f"{user_profile[f'{channel}_days'].gt(0).sum():,}"
    )

print("\nVALIDATION: PASS")
print("No files created or modified.")


NOTEBOOK 02 — DATA LOADING & INTEGRITY
Population users       : 3,995
Future-active users    : 3,737
Users without Future   : 258
Eligible Future days   : 387,162
Profile features       : 101

DATA COVERAGE
------------------------------------------------------------------------
First observed day     : 2011-01-03
Last observed day      : 2011-06-01

CHANNEL COVERAGE
------------------------------------------------------------------------
EMAIL    active users: 3,737
USB      active users: 719
HTTP     active users: 567

VALIDATION: PASS
No files created or modified.



## 2. Temporal Availability and Calendar Gaps

### 2.1 Eligible Workday Continuity

**Objective:** Examine the continuity of each user's eligible Future workdays before discovering candidate attack windows.

**This analysis will:**
- Identify the first and last eligible Future day for each user.
- Calculate the number of calendar weekdays between these dates, inclusive.
- Count missing calendar weekdays within each user's observed Future span.
- Count calendar interruptions and measure their lengths.
- Split each user's eligible workdays into contiguous calendar-weekday segments.
- Measure the number and lengths of these segments.
- Calculate the percentage of available calendar weekdays that are eligible.

**Definitions:**
- **Calendar weekday:** Monday through Friday, excluding Saturday and Sunday. Public holidays are not separately excluded.
- **Eligible workday:** A Future user-day meeting the original eligibility criteria.
- **Calendar interruption:** One or more calendar weekdays without an eligible record between two observed eligible days.
- **Contiguous segment:** A sequence of eligible dates without missing calendar weekdays.
- **Observed span:** The inclusive period between the user's first and last eligible Future dates.

**Important:**
- Days before the first or after the last eligible record are not counted as internal gaps.
- A calendar interruption does not automatically invalidate an attack window.
- No attack duration, channel sequence, or injection rule is imposed.
- Results remain in memory; no files are saved in this section.



In [4]:

# ============================================================
# 2.1 ELIGIBLE WORKDAY CONTINUITY
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PREPARE DAILY CALENDAR
# ------------------------------------------------------------

calendar_daily = future_daily[
    ["user", "day"]
].copy()

calendar_daily["day"] = pd.to_datetime(
    calendar_daily["day"]
)

calendar_daily = (
    calendar_daily
    .sort_values(["user", "day"])
    .reset_index(drop=True)
)

assert not calendar_daily.duplicated(["user", "day"]).any()
assert calendar_daily["day"].dt.dayofweek.lt(5).all()

previous_day = (
    calendar_daily.groupby("user")["day"].shift()
)

has_previous = previous_day.notna()

# Count missing Monday-Friday dates between
# consecutive eligible records of the same user.
missing_weekdays = np.zeros(
    len(calendar_daily),
    dtype=np.int32
)

missing_weekdays[has_previous.to_numpy()] = (
    np.busday_count(
        previous_day.loc[
            has_previous
        ].to_numpy(dtype="datetime64[D]"),
        calendar_daily.loc[
            has_previous, "day"
        ].to_numpy(dtype="datetime64[D]")
    ) - 1
)

assert (missing_weekdays >= 0).all()

calendar_daily["missing_weekdays_before"] = (
    missing_weekdays
)

# ------------------------------------------------------------
# 2. IDENTIFY CONTIGUOUS CALENDAR SEGMENTS
# ------------------------------------------------------------

calendar_daily["new_segment"] = (
    previous_day.isna()
    | calendar_daily["missing_weekdays_before"].gt(0)
)

calendar_daily["segment_id"] = (
    calendar_daily.groupby("user")["new_segment"]
    .cumsum()
    .astype(int)
)

calendar_segments = (
    calendar_daily
    .groupby(["user", "segment_id"])
    .agg(
        segment_start=("day", "min"),
        segment_end=("day", "max"),
        segment_eligible_days=("day", "size")
    )
    .reset_index()
)

segment_summary = (
    calendar_segments.groupby("user")
    .agg(
        calendar_segment_count=("segment_id", "size"),
        longest_contiguous_eligible_run=(
            "segment_eligible_days", "max"
        ),
        median_contiguous_eligible_run=(
            "segment_eligible_days", "median"
        )
    )
)

# ------------------------------------------------------------
# 3. USER-LEVEL CALENDAR STATISTICS
# ------------------------------------------------------------

calendar_stats = (
    calendar_daily.groupby("user")
    .agg(
        first_eligible_day=("day", "min"),
        last_eligible_day=("day", "max"),
        eligible_day_rows=("day", "size"),
        missing_calendar_weekdays_inside_span=(
            "missing_weekdays_before", "sum"
        ),
        calendar_gap_events=(
            "missing_weekdays_before",
            lambda x: int(x.gt(0).sum())
        ),
        longest_missing_weekday_gap=(
            "missing_weekdays_before", "max"
        )
    )
)

# Count Monday-Friday dates inclusively.
calendar_stats["calendar_span_weekdays"] = (
    np.busday_count(
        calendar_stats[
            "first_eligible_day"
        ].to_numpy(dtype="datetime64[D]"),
        calendar_stats[
            "last_eligible_day"
        ].to_numpy(dtype="datetime64[D]")
    ) + 1
)

calendar_stats = calendar_stats.join(
    segment_summary
)

# ------------------------------------------------------------
# 4. INCLUDE ALL USERS
# ------------------------------------------------------------

temporal_availability = (
    user_profile[
        ["user", "future_workdays"]
    ]
    .copy()
    .reset_index(drop=True)
)

for column in calendar_stats.columns:
    temporal_availability[column] = (
        temporal_availability["user"]
        .map(calendar_stats[column])
    )

count_columns = [
    "eligible_day_rows",
    "missing_calendar_weekdays_inside_span",
    "calendar_gap_events",
    "longest_missing_weekday_gap",
    "calendar_span_weekdays",
    "calendar_segment_count",
    "longest_contiguous_eligible_run"
]

for column in count_columns:
    temporal_availability[column] = (
        temporal_availability[column]
        .fillna(0)
        .astype(int)
    )

temporal_availability["calendar_coverage_pct"] = (
    100
    * temporal_availability["future_workdays"]
    / temporal_availability[
        "calendar_span_weekdays"
    ].replace(0, np.nan)
)

# ------------------------------------------------------------
# 5. INTEGRITY VALIDATION
# ------------------------------------------------------------

assert len(temporal_availability) == len(user_profile)

# User-day coverage
assert (
    temporal_availability["eligible_day_rows"]
    == temporal_availability["future_workdays"]
).all()

# Calendar span must equal eligible days plus internal gaps.
assert (
    temporal_availability["calendar_span_weekdays"]
    == (
        temporal_availability["future_workdays"]
        + temporal_availability[
            "missing_calendar_weekdays_inside_span"
        ]
    )
).all()

# Every eligible day must belong to exactly one segment.
assert (
    calendar_segments["segment_eligible_days"].sum()
    == len(future_daily)
)

# Segment count must equal gap events + 1
# for users who have eligible Future days.
available_mask = (
    temporal_availability["future_workdays"] > 0
)

assert (
    temporal_availability.loc[
        available_mask, "calendar_segment_count"
    ]
    == (
        temporal_availability.loc[
            available_mask, "calendar_gap_events"
        ] + 1
    )
).all()

# Longest contiguous run cannot exceed total eligible days.
assert (
    temporal_availability["longest_contiguous_eligible_run"]
    <= temporal_availability["future_workdays"]
).all()

# ------------------------------------------------------------
# 6. CROSS-CHECK WITH NOTEBOOK 01
# ------------------------------------------------------------

# Verify longest eligible calendar-weekday streak.
assert np.array_equal(
    temporal_availability[
        "longest_contiguous_eligible_run"
    ].to_numpy(),
    user_profile[
        "max_eligible_calendar_weekday_streak"
    ].fillna(0).astype(int).to_numpy()
), "Mismatch with Notebook 01 calendar streaks"

# Compare date VALUES rather than pandas Series metadata.
# Both sides are converted to calendar-date precision.
def verify_date_column(actual_column, expected_column, label):

    actual = pd.to_datetime(
        temporal_availability[actual_column]
    ).to_numpy(dtype="datetime64[D]")

    expected = pd.to_datetime(
        user_profile[expected_column]
    ).to_numpy(dtype="datetime64[D]")

    # NaT must match NaT for users without Future workdays.
    matches = (
        (actual == expected)
        | (np.isnat(actual) & np.isnat(expected))
    )

    mismatch_count = int((~matches).sum())

    if mismatch_count > 0:

        mismatches = pd.DataFrame({
            "user": temporal_availability.loc[
                ~matches, "user"
            ].to_numpy(),
            "notebook_02": actual[~matches],
            "notebook_01": expected[~matches]
        })

        print(f"\n{label}: {mismatch_count} mismatches")
        print(mismatches.head(10).to_string(index=False))

        raise AssertionError(
            f"{label}: Actual date values differ"
        )

    return mismatch_count


first_date_mismatches = verify_date_column(
    "first_eligible_day",
    "future_first_eligible_day",
    "First eligible day"
)

last_date_mismatches = verify_date_column(
    "last_eligible_day",
    "future_last_eligible_day",
    "Last eligible day"
)

# ------------------------------------------------------------
# 7. USER-LEVEL DISTRIBUTIONS
# ------------------------------------------------------------

available_users = temporal_availability.loc[
    temporal_availability["future_workdays"] > 0
]

summary_columns = [
    "future_workdays",
    "calendar_span_weekdays",
    "missing_calendar_weekdays_inside_span",
    "calendar_gap_events",
    "longest_missing_weekday_gap",
    "calendar_segment_count",
    "longest_contiguous_eligible_run",
    "calendar_coverage_pct"
]

summary = (
    available_users[summary_columns]
    .quantile([0.25, 0.50, 0.75, 0.90, 1.00])
    .T
    .round(2)
)

summary.columns = [
    "P25", "Median", "P75", "P90", "Max"
]

# ------------------------------------------------------------
# 8. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 72)
print("TEMPORAL AVAILABILITY — FUTURE PERIOD")
print("=" * 72)

print(f"Total users                    : {len(user_profile):,}")
print(f"Users with eligible days       : {len(available_users):,}")

print(
    "Users with calendar gaps       :",
    f"{available_users['calendar_gap_events'].gt(0).sum():,}"
)

print(
    "Users with no internal gaps    :",
    f"{available_users['calendar_gap_events'].eq(0).sum():,}"
)

print(
    "Total calendar gap events      :",
    f"{available_users['calendar_gap_events'].sum():,}"
)

print(
    "Total missing calendar weekdays:",
    f"{available_users['missing_calendar_weekdays_inside_span'].sum():,}"
)

print(
    "Total contiguous segments      :",
    f"{len(calendar_segments):,}"
)

print("\nUSER-LEVEL DISTRIBUTIONS")
print("-" * 72)
print(summary.to_string())

# ------------------------------------------------------------
# 9. SHARED CALENDAR INTERRUPTIONS
# ------------------------------------------------------------

gap_resumptions = (
    calendar_daily.loc[
        calendar_daily["missing_weekdays_before"] > 0
    ]
    .groupby("day")
    .agg(
        users_resuming=("user", "nunique"),
        missing_weekdays_total=(
            "missing_weekdays_before", "sum"
        )
    )
    .sort_values(
        "users_resuming",
        ascending=False
    )
    .head(6)
)

print("\nMOST COMMON GAP-RESUMPTION DATES")
print("-" * 72)
print(gap_resumptions.to_string())

print("\nCROSS-NOTEBOOK DATE VALIDATION")
print("-" * 72)
print(f"First-date mismatches : {first_date_mismatches}")
print(f"Last-date mismatches  : {last_date_mismatches}")

print("\nVALIDATION: PASS")
print("No files saved.")


TEMPORAL AVAILABILITY — FUTURE PERIOD
Total users                    : 3,995
Users with eligible days       : 3,737
Users with calendar gaps       : 3,570
Users with no internal gaps    : 167
Total calendar gap events      : 6,976
Total missing calendar weekdays: 6,976
Total contiguous segments      : 10,713

USER-LEVEL DISTRIBUTIONS
------------------------------------------------------------------------
                                          P25  Median     P75     P90    Max
future_workdays                        105.00  105.00  105.00  105.00  108.0
calendar_span_weekdays                 107.00  107.00  107.00  107.00  108.0
missing_calendar_weekdays_inside_span    2.00    2.00    2.00    2.00    3.0
calendar_gap_events                      2.00    2.00    2.00    2.00    3.0
longest_missing_weekday_gap              1.00    1.00    1.00    1.00    1.0
calendar_segment_count                   3.00    3.00    3.00    3.00    4.0
longest_contiguous_eligible_run         79.00   79.0


## 3. Candidate Window Discovery

### 3.1 User-Level Window Capacity

**Objective:** Discover all possible window lengths and quantify the number of available windows for each user based on actual eligible Future workdays.

### Window Definitions

**Eligible-Workday Window**
- Contains consecutive eligible workday records for the same user.
- May span calendar weekdays without eligible records.
- Its length is measured in eligible workdays.

**Calendar-Continuous Window**
- Contains consecutive eligible workdays with no missing Monday–Friday dates.
- It is a subset of Eligible-Workday Windows.

### Analysis

For every user and every naturally supported window length:

- Count all possible eligible-workday windows.
- Count calendar-continuous windows.
- Count windows crossing one or more ineligible calendar weekdays.
- Identify the maximum supported window length.
- Compare population coverage across window lengths.
- Identify how calendar interruptions affect window availability.

### Methodology

Window lengths are derived from the actual number of eligible Future workdays, rather than predefined attack durations.

At this stage, a window is considered temporally available when its required eligible workday records exist.

**Important:**
- Temporal availability does not imply channel-pattern feasibility.
- No attack scenarios or switching sequences are assigned.
- No minimum window length is imposed.
- Overlapping windows are counted as separate candidates, not independent attack opportunities.
- Historical Baseline and original Future datasets remain unchanged.

### Expected In-Memory Results

- `window_capacity_by_user`
- `window_length_capacity`
- `window_length_summary`

No output files are saved in this section.


In [5]:

# ============================================================
# 3.1 USER-LEVEL WINDOW CAPACITY DISCOVERY
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. PREPARE USERS AND OBSERVED WINDOW LENGTHS
# ------------------------------------------------------------

user_ids = temporal_availability["user"].to_numpy()

user_workdays = (
    temporal_availability["future_workdays"]
    .to_numpy(dtype=np.int64)
)

max_length = int(user_workdays.max())

# All naturally supported lengths, not predefined scenarios.
window_lengths = np.arange(
    1, max_length + 1, dtype=np.int64
)

# ------------------------------------------------------------
# 2. ALL ELIGIBLE-WORKDAY WINDOWS
# ------------------------------------------------------------

# For n eligible workdays, a window of length L
# has max(n - L + 1, 0) possible starting positions.

eligible_window_counts = np.maximum(
    user_workdays[:, None]
    - window_lengths[None, :]
    + 1,
    0
)

# ------------------------------------------------------------
# 3. CALENDAR-CONTINUOUS WINDOWS
# ------------------------------------------------------------

# Build a histogram of contiguous segment lengths
# for each user.

user_positions = pd.Index(user_ids).get_indexer(
    calendar_segments["user"]
)

segment_lengths = (
    calendar_segments["segment_eligible_days"]
    .to_numpy(dtype=np.int64)
)

assert (user_positions >= 0).all()
assert (segment_lengths >= 1).all()
assert (segment_lengths <= max_length).all()

segment_histogram = np.zeros(
    (len(user_ids), max_length),
    dtype=np.int64
)

np.add.at(
    segment_histogram,
    (user_positions, segment_lengths - 1),
    1
)

# For each length L:
# First count segments with at least L days.
segments_at_least_length = (
    np.cumsum(
        segment_histogram[:, ::-1],
        axis=1
    )[:, ::-1]
)

# Then count all windows within those segments.
calendar_window_counts = (
    np.cumsum(
        segments_at_least_length[:, ::-1],
        axis=1
    )[:, ::-1]
)

# ------------------------------------------------------------
# 4. WINDOWS CROSSING CALENDAR GAPS
# ------------------------------------------------------------

cross_gap_window_counts = (
    eligible_window_counts
    - calendar_window_counts
)

# ------------------------------------------------------------
# 5. USER-LEVEL WINDOW CAPACITY
# ------------------------------------------------------------

window_capacity_by_user = temporal_availability[
    [
        "user",
        "future_workdays",
        "calendar_segment_count",
        "longest_contiguous_eligible_run",
        "calendar_gap_events"
    ]
].copy().reset_index(drop=True)

window_capacity_by_user[
    "max_eligible_window_length"
] = user_workdays

window_capacity_by_user[
    "max_calendar_contiguous_window_length"
] = (
    temporal_availability[
        "longest_contiguous_eligible_run"
    ].to_numpy(dtype=np.int64)
)

window_capacity_by_user[
    "all_eligible_windows"
] = eligible_window_counts.sum(axis=1)

window_capacity_by_user[
    "all_calendar_contiguous_windows"
] = calendar_window_counts.sum(axis=1)

window_capacity_by_user[
    "all_cross_gap_windows"
] = cross_gap_window_counts.sum(axis=1)

window_capacity_by_user[
    "supported_eligible_window_lengths"
] = (eligible_window_counts > 0).sum(axis=1)

window_capacity_by_user[
    "supported_calendar_contiguous_lengths"
] = (calendar_window_counts > 0).sum(axis=1)

# ------------------------------------------------------------
# 6. USER × WINDOW LENGTH CAPACITY
# ------------------------------------------------------------

# Keep only lengths supported by each user.
# This is NOT an enumeration of individual start/end windows.

user_idx, length_idx = np.nonzero(
    eligible_window_counts > 0
)

window_length_capacity = pd.DataFrame({
    "user": user_ids[user_idx],
    "window_workdays": window_lengths[length_idx],
    "eligible_windows": (
        eligible_window_counts[user_idx, length_idx]
    ),
    "calendar_contiguous_windows": (
        calendar_window_counts[user_idx, length_idx]
    ),
    "cross_gap_windows": (
        cross_gap_window_counts[user_idx, length_idx]
    )
})

window_length_capacity[
    "has_calendar_contiguous_option"
] = (
    window_length_capacity[
        "calendar_contiguous_windows"
    ] > 0
)

# ------------------------------------------------------------
# 7. POPULATION COVERAGE BY WINDOW LENGTH
# ------------------------------------------------------------

window_length_summary = (
    window_length_capacity
    .groupby("window_workdays")
    .agg(
        users_with_eligible_windows=("user", "nunique"),
        total_eligible_windows=("eligible_windows", "sum"),
        total_calendar_contiguous_windows=(
            "calendar_contiguous_windows", "sum"
        ),
        total_cross_gap_windows=("cross_gap_windows", "sum"),
        users_with_contiguous_windows=(
            "has_calendar_contiguous_option", "sum"
        )
    )
    .reset_index()
)

window_length_summary[
    "users_without_contiguous_option"
] = (
    window_length_summary[
        "users_with_eligible_windows"
    ]
    - window_length_summary[
        "users_with_contiguous_windows"
    ]
)

# ------------------------------------------------------------
# 8. COMPREHENSIVE VALIDATION
# ------------------------------------------------------------

assert len(user_ids) == len(user_profile) == 3995
assert user_workdays.sum() == len(future_daily) == 387162

assert (
    segment_histogram * window_lengths[None, :]
).sum() == len(future_daily)

assert np.array_equal(
    (
        segment_histogram
        * window_lengths[None, :]
    ).sum(axis=1),
    user_workdays
)

assert np.array_equal(
    eligible_window_counts[:, 0],
    user_workdays
)

assert np.array_equal(
    calendar_window_counts[:, 0],
    user_workdays
)

assert (calendar_window_counts <= eligible_window_counts).all()
assert (cross_gap_window_counts >= 0).all()

# All possible eligible windows across all lengths
# equal n(n+1)/2 for each user.
assert np.array_equal(
    eligible_window_counts.sum(axis=1),
    user_workdays * (user_workdays + 1) // 2
)

# Calendar-contiguous windows across all lengths
# must equal the sum of segment-based combinations.
segment_window_totals = (
    segment_lengths * (segment_lengths + 1) // 2
)

assert (
    calendar_window_counts.sum()
    == segment_window_totals.sum()
)

# For length 2, each internal gap removes
# exactly one calendar-contiguous adjacent pair.
if max_length >= 2:
    assert (
        cross_gap_window_counts[:, 1].sum()
        == temporal_availability[
            "calendar_gap_events"
        ].sum()
    )

assert (
    len(window_length_capacity)
    == len(future_daily)
)

assert not window_length_capacity.duplicated(
    ["user", "window_workdays"]
).any()

assert (
    window_length_capacity[
        "eligible_windows"
    ]
    == (
        window_length_capacity[
            "calendar_contiguous_windows"
        ]
        + window_length_capacity[
            "cross_gap_windows"
        ]
    )
).all()

# ------------------------------------------------------------
# 9. CONSOLIDATED RESULTS
# ------------------------------------------------------------

available_capacity = window_capacity_by_user.loc[
    window_capacity_by_user["future_workdays"] > 0
]

print("=" * 76)
print("CANDIDATE WINDOW CAPACITY — FUTURE PERIOD")
print("=" * 76)

print(f"Population users                  : {len(user_ids):,}")
print(f"Users with eligible Future days   : {len(available_capacity):,}")
print(f"Observed window lengths           : 1 to {max_length}")
print(f"User-length capacity records      : {len(window_length_capacity):,}")

print(
    "All eligible window candidates    :",
    f"{eligible_window_counts.sum():,}"
)

print(
    "All calendar-contiguous candidates:",
    f"{calendar_window_counts.sum():,}"
)

print(
    "Candidates crossing calendar gaps :",
    f"{cross_gap_window_counts.sum():,}"
)

# ------------------------------------------------------------
# 10. USER-LEVEL DISTRIBUTIONS
# ------------------------------------------------------------

print("\nUSER-LEVEL WINDOW CAPACITY")
print("-" * 76)

user_metrics = [
    "max_eligible_window_length",
    "max_calendar_contiguous_window_length",
    "all_eligible_windows",
    "all_calendar_contiguous_windows",
    "all_cross_gap_windows"
]

user_summary = (
    available_capacity[user_metrics]
    .quantile([0.25, 0.50, 0.75, 0.90, 1.00])
    .T
)

user_summary.columns = [
    "P25", "Median", "P75", "P90", "Max"
]

print(user_summary.round(2).to_string())

# ------------------------------------------------------------
# 11. POPULATION COVERAGE — ILLUSTRATIVE LENGTHS
# ------------------------------------------------------------

# These are reporting checkpoints only.
# They are NOT attack durations or eligibility thresholds.

checkpoints = sorted(
    set([
        1, 5, 10, 20, 40, 60,
        79, 80, 100, max_length
    ])
)

checkpoints = [
    length for length in checkpoints
    if length <= max_length
]

checkpoint_summary = (
    window_length_summary.loc[
        window_length_summary[
            "window_workdays"
        ].isin(checkpoints),
        [
            "window_workdays",
            "users_with_eligible_windows",
            "users_with_contiguous_windows",
            "users_without_contiguous_option",
            "total_eligible_windows",
            "total_calendar_contiguous_windows",
            "total_cross_gap_windows"
        ]
    ]
)

print("\nWINDOW LENGTH COVERAGE — EXPLORATORY CHECKPOINTS")
print("-" * 76)
print(checkpoint_summary.to_string(index=False))

print("\nVALIDATION: PASS")
print("No scenario length or switching pattern imposed.")
print("No files saved.")


CANDIDATE WINDOW CAPACITY — FUTURE PERIOD
Population users                  : 3,995
Users with eligible Future days   : 3,737
Observed window lengths           : 1 to 108
User-length capacity records      : 387,162
All eligible window candidates    : 20,436,421
All calendar-contiguous candidates: 13,158,206
Candidates crossing calendar gaps : 7,278,215

USER-LEVEL WINDOW CAPACITY
----------------------------------------------------------------------------
                                          P25  Median     P75     P90     Max
max_eligible_window_length              105.0   105.0   105.0   105.0   108.0
max_calendar_contiguous_window_length    79.0    79.0    79.0    79.0   108.0
all_eligible_windows                   5565.0  5565.0  5565.0  5565.0  5886.0
all_calendar_contiguous_windows        3486.0  3486.0  3486.0  3486.0  5886.0
all_cross_gap_windows                  2079.0  2079.0  2079.0  2079.0  3570.0

WINDOW LENGTH COVERAGE — EXPLORATORY CHECKPOINTS
----------------------


### 3.2 Channel Availability Within Candidate Windows

**Objective:** Examine which target channels are naturally observed within every temporally available window.

**Method:**
- Evaluate every eligible-workday window for each user.
- Identify whether Email, USB, and HTTP Upload appear at least once within the window.
- Classify windows by their exact observed channel set.
- Count windows containing zero, one, two, or three distinct target channels.
- Summarize population coverage across all supported window lengths.

**Window Channel States:**
- NONE: No target-channel activity observed.
- E, U, H: Only one target channel observed.
- EU, EH, UH: Two distinct channels observed.
- EUH: All three channels observed.

**Interpretation:**
- Channel presence refers to activity anywhere within the window, not necessarily on the same day.
- The order of channel activity is not evaluated yet.
- Natural channel presence provides behavioral context; it is not an injection eligibility requirement.
- No attack scenario, switching sequence, or minimum channel count is imposed.

**Implementation:**
- Calculate window-level channel presence using cumulative daily activity counts.
- Aggregate results by user and window length.
- Avoid retaining millions of individual window records.

**Expected Results:**
- `channel_window_capacity`
- `window_channel_summary`

No files are saved in this section.


In [6]:

# ============================================================
# 3.2 CHANNEL AVAILABILITY WITHIN CANDIDATE WINDOWS
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

channel_flags = [
    "email_active",
    "usb_active",
    "http_active"
]

# Bit encoding: E=1, U=2, H=4
state_names = {
    0: "NONE",
    1: "E",
    2: "U",
    3: "EU",
    4: "H",
    5: "EH",
    6: "UH",
    7: "EUH"
}

state_cols = {
    code: f"windows_{name}"
    for code, name in state_names.items()
}

state_weights = np.array(
    [1, 2, 4], dtype=np.int16
)

# ------------------------------------------------------------
# 2. COUNT CHANNEL STATES IN ALL WINDOWS
# ------------------------------------------------------------

capacity_parts = []

for user, group in future_daily.groupby(
    "user", sort=False
):

    flags = group[channel_flags].to_numpy(
        dtype=np.int16
    )

    n = len(flags)

    if n == 0:
        continue

    # Every start/end pair representing a valid window.
    # Arrays are created for one user at a time.
    start_idx, end_idx = np.triu_indices(n)

    lengths = end_idx - start_idx + 1

    # Prefix sums allow exact window activity queries.
    prefix = np.vstack([
        np.zeros((1, 3), dtype=np.int32),
        np.cumsum(flags, axis=0, dtype=np.int32)
    ])

    # Number of active days per channel within each window.
    channel_days_in_window = (
        prefix[end_idx + 1] - prefix[start_idx]
    )

    # Which channels appear at least once in the window?
    present = channel_days_in_window > 0

    observed_state = (
        present.astype(np.int16) @ state_weights
    )

    # Count the eight states separately for every length.
    codes = (
        (lengths - 1) * 8 + observed_state
    )

    counts = np.bincount(
        codes,
        minlength=n * 8
    ).reshape(n, 8)

    # At each length L, there must be n-L+1 windows.
    expected = n - np.arange(n)

    assert np.array_equal(
        counts.sum(axis=1),
        expected
    ), f"Window count mismatch: {user}"

    part = pd.DataFrame({
        "user": [user] * n,
        "window_workdays": np.arange(1, n + 1),
        "eligible_windows": expected
    })

    for code, col in state_cols.items():
        part[col] = counts[:, code]

    capacity_parts.append(part)

channel_window_capacity = pd.concat(
    capacity_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. DERIVED WINDOW CHARACTERISTICS
# ------------------------------------------------------------

two_channel_states = [
    "windows_EU",
    "windows_EH",
    "windows_UH"
]

channel_window_capacity[
    "windows_one_observed_channel"
] = channel_window_capacity[
    ["windows_E", "windows_U", "windows_H"]
].sum(axis=1)

channel_window_capacity[
    "windows_two_observed_channels"
] = channel_window_capacity[
    two_channel_states
].sum(axis=1)

channel_window_capacity[
    "windows_three_observed_channels"
] = channel_window_capacity["windows_EUH"]

channel_window_capacity[
    "windows_2plus_observed_channels"
] = (
    channel_window_capacity[
        "windows_two_observed_channels"
    ]
    + channel_window_capacity[
        "windows_three_observed_channels"
    ]
)

channel_window_capacity[
    "windows_any_observed_channel"
] = (
    channel_window_capacity["eligible_windows"]
    - channel_window_capacity["windows_NONE"]
)

# ------------------------------------------------------------
# 4. VALIDATE AGAINST TEMPORAL WINDOW CAPACITY
# ------------------------------------------------------------

comparison = channel_window_capacity[
    ["user", "window_workdays", "eligible_windows"]
].merge(
    window_length_capacity[
        ["user", "window_workdays", "eligible_windows"]
    ],
    on=["user", "window_workdays"],
    how="outer",
    validate="one_to_one",
    indicator=True,
    suffixes=("_channel", "_temporal")
)

assert comparison["_merge"].eq("both").all()

assert (
    comparison["eligible_windows_channel"]
    == comparison["eligible_windows_temporal"]
).all()

assert len(channel_window_capacity) == len(
    window_length_capacity
)

assert (
    channel_window_capacity[
        list(state_cols.values())
    ].sum(axis=1)
    == channel_window_capacity["eligible_windows"]
).all()

assert (
    channel_window_capacity[
        "windows_2plus_observed_channels"
    ]
    <= channel_window_capacity[
        "windows_any_observed_channel"
    ]
).all()

# ------------------------------------------------------------
# 5. CROSS-CHECK ONE-DAY WINDOWS WITH NOTEBOOK 01
# ------------------------------------------------------------

single_day_windows = channel_window_capacity.loc[
    channel_window_capacity["window_workdays"] == 1
]

expected_single_day = {
    "NONE": int(
        (user_profile["future_workdays"]
         - user_profile["any_channel_days"]).sum()
    )
}

for state in ["E", "U", "H", "EU", "EH", "UH", "EUH"]:
    expected_single_day[state] = int(
        user_profile[f"days_{state}"].sum()
    )

for state, expected_count in expected_single_day.items():
    actual_count = int(
        single_day_windows[f"windows_{state}"].sum()
    )

    assert actual_count == expected_count, (
        f"One-day channel mismatch: {state}"
    )

# ------------------------------------------------------------
# 6. POPULATION SUMMARY BY WINDOW LENGTH
# ------------------------------------------------------------

window_channel_summary = (
    channel_window_capacity
    .groupby("window_workdays")
    .agg(
        users_with_windows=("user", "size"),
        total_windows=("eligible_windows", "sum"),
        windows_without_target_activity=(
            "windows_NONE", "sum"
        ),
        windows_with_any_channel=(
            "windows_any_observed_channel", "sum"
        ),
        windows_with_2plus_channels=(
            "windows_2plus_observed_channels", "sum"
        ),
        windows_with_3_channels=(
            "windows_EUH", "sum"
        ),
        users_with_2plus_channel_option=(
            "windows_2plus_observed_channels",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_3_channel_option=(
            "windows_EUH",
            lambda x: int(x.gt(0).sum())
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. FINAL VALIDATION
# ------------------------------------------------------------

assert (
    window_channel_summary["total_windows"].sum()
    == window_length_capacity[
        "eligible_windows"
    ].sum()
)

assert (
    window_channel_summary[
        "windows_without_target_activity"
    ]
    + window_channel_summary[
        "windows_with_any_channel"
    ]
    == window_channel_summary["total_windows"]
).all()

assert (
    window_channel_summary[
        "windows_with_3_channels"
    ]
    <= window_channel_summary[
        "windows_with_2plus_channels"
    ]
).all()

# ------------------------------------------------------------
# 8. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 76)
print("CHANNEL AVAILABILITY WITHIN CANDIDATE WINDOWS")
print("=" * 76)

print(
    "User-length records:",
    f"{len(channel_window_capacity):,}"
)

print(
    "Total windows:",
    f"{channel_window_capacity['eligible_windows'].sum():,}"
)

print("\nONE-DAY WINDOW STATES")
print("-" * 76)

for state in state_names.values():
    count = int(
        single_day_windows[f"windows_{state}"].sum()
    )
    print(f"{state:<5}: {count:,}")

# Reporting checkpoints only, not attack-duration rules.
checkpoints = [
    n for n in [1, 5, 10, 20, 40, 60, 79, 80, 100, 108]
    if n <= int(future_daily.groupby("user").size().max())
]

selected = window_channel_summary.loc[
    window_channel_summary["window_workdays"].isin(
        checkpoints
    ),
    [
        "window_workdays",
        "users_with_windows",
        "users_with_2plus_channel_option",
        "users_with_3_channel_option",
        "total_windows",
        "windows_with_2plus_channels",
        "windows_with_3_channels"
    ]
]

print("\nCHANNEL COVERAGE — EXPLORATORY LENGTHS")
print("-" * 76)
print(selected.to_string(index=False))

print("\nVALIDATION: PASS")
print("No channel sequence or scenario imposed.")
print("No files saved.")


CHANNEL AVAILABILITY WITHIN CANDIDATE WINDOWS
User-length records: 387,162
Total windows: 20,436,421

ONE-DAY WINDOW STATES
----------------------------------------------------------------------------
NONE : 147,442
E    : 198,212
U    : 11,822
EU   : 15,788
H    : 5,061
EH   : 6,429
UH   : 998
EUH  : 1,410

CHANNEL COVERAGE — EXPLORATORY LENGTHS
----------------------------------------------------------------------------
 window_workdays  users_with_windows  users_with_2plus_channel_option  users_with_3_channel_option  total_windows  windows_with_2plus_channels  windows_with_3_channels
               1                3737                              985                          239         387162                        24625                     1410
               5                3736                              995                          290         372217                        74997                    12330
              10                3728                              993 


### 3.3 Distinct-Day Channel Support

**Objective:** Determine whether channels observed within each candidate window can be represented on separate eligible workdays.

**This analysis distinguishes:**
- **Observed channel presence:** Two or three channels appear somewhere within a window.
- **Two-channel distinct-day support:** At least two different channels can be assigned to two different naturally active days.
- **Three-channel distinct-day support:** Email, USB, and HTTP can each be assigned to a different naturally active day.

**Methodology:**
- Evaluate all eligible-workday windows without imposing specific lengths.
- Use the exact daily channel combinations already present in Future.
- Account for same-day multi-channel overlap.
- Verify three-channel support using distinct-day assignment conditions, including cases where multiple channels share the same active days.
- Aggregate results by user and window length without storing individual windows.

**Interpretation:**
- Distinct-day support is an observed behavioral property, not a final injection feasibility decision.
- No fixed channel order is imposed.
- No natural switching count is required.
- Attack volume, stealth limits, Off-days, and scenario assignment remain outside this analysis.
- All source datasets remain read-only.

**Expected Outputs (in memory):**
- `distinct_day_window_capacity`
- `distinct_day_window_summary`


In [7]:

# ============================================================
# 3.3 DISTINCT-DAY CHANNEL SUPPORT
# ============================================================

import numpy as np
import pandas as pd

flags_cols = ["email_active", "usb_active", "http_active"]
weights = np.array([1, 2, 4], dtype=np.int32)

results = []

# ------------------------------------------------------------
# 1. EXAMINE EVERY WINDOW, ONE USER AT A TIME
# ------------------------------------------------------------

for user, group in future_daily.groupby("user", sort=False):

    flags = group[flags_cols].to_numpy(dtype=np.int32)
    n = len(flags)

    if n == 0:
        continue

    # Daily state: E=1, U=2, H=4.
    daily_state = flags @ weights

    # Prefix columns:
    # E days, U days, H days, any active days,
    # E-only days, U-only days, H-only days.
    daily_metrics = np.column_stack([
        flags,
        (daily_state > 0).astype(np.int32),
        (daily_state == 1).astype(np.int32),
        (daily_state == 2).astype(np.int32),
        (daily_state == 4).astype(np.int32)
    ])

    prefix = np.vstack([
        np.zeros((1, 7), dtype=np.int32),
        np.cumsum(daily_metrics, axis=0, dtype=np.int32)
    ])

    start, end = np.triu_indices(n)
    lengths = end - start + 1

    counts = prefix[end + 1] - prefix[start]

    channel_counts = counts[:, :3]

    any_active_days = counts[:, 3]
    e_only_days = counts[:, 4]
    u_only_days = counts[:, 5]
    h_only_days = counts[:, 6]

    distinct_channels = (
        channel_counts > 0
    ).sum(axis=1)

    # --------------------------------------------------------
    # 2. TWO DIFFERENT CHANNELS ON TWO DISTINCT DAYS
    # --------------------------------------------------------

    observed_2plus = distinct_channels >= 2

    distinct_day_2plus = (
        observed_2plus
        & (any_active_days >= 2)
    )

    same_day_only_2plus = (
        observed_2plus
        & (any_active_days == 1)
    )

    # --------------------------------------------------------
    # 3. THREE CHANNELS ON THREE DISTINCT DAYS
    # --------------------------------------------------------

    observed_three = distinct_channels == 3

    # For three distinct-day channel assignments:
    # 1. All channels must appear.
    # 2. At least three active days must exist.
    # 3. Every pair of channels must collectively
    #    have activity on at least two distinct days.
    #
    # Example:
    # E+U days = all active days minus H-only days.

    eu_union_days = any_active_days - h_only_days
    eh_union_days = any_active_days - u_only_days
    uh_union_days = any_active_days - e_only_days

    distinct_day_three = (
        observed_three
        & (any_active_days >= 3)
        & (eu_union_days >= 2)
        & (eh_union_days >= 2)
        & (uh_union_days >= 2)
    )

    three_without_assignment = (
        observed_three & ~distinct_day_three
    )

    # Three or more active days are present,
    # but all three channels still cannot be
    # assigned to three distinct days.
    three_pair_constraint_fail = (
        observed_three
        & (any_active_days >= 3)
        & ~distinct_day_three
    )

    # --------------------------------------------------------
    # 4. AGGREGATE BY WINDOW LENGTH
    # --------------------------------------------------------

    def count_by_length(mask):
        return np.bincount(
            lengths[mask] - 1,
            minlength=n
        ).astype(np.int64)

    part = pd.DataFrame({
        "user": user,
        "window_workdays": np.arange(1, n + 1),
        "observed_2plus_windows":
            count_by_length(observed_2plus),
        "distinct_day_2plus_windows":
            count_by_length(distinct_day_2plus),
        "same_day_only_2plus_windows":
            count_by_length(same_day_only_2plus),
        "observed_three_windows":
            count_by_length(observed_three),
        "distinct_day_three_windows":
            count_by_length(distinct_day_three),
        "three_without_assignment_windows":
            count_by_length(three_without_assignment),
        "three_pair_constraint_fail_windows":
            count_by_length(three_pair_constraint_fail)
    })

    results.append(part)

distinct_day_window_capacity = pd.concat(
    results,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. CROSS-CHECK WITH SECTION 3.2
# ------------------------------------------------------------

reference = channel_window_capacity[
    [
        "user",
        "window_workdays",
        "eligible_windows",
        "windows_2plus_observed_channels",
        "windows_EUH"
    ]
]

distinct_day_window_capacity = (
    distinct_day_window_capacity.merge(
        reference,
        on=["user", "window_workdays"],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert distinct_day_window_capacity["_merge"].eq("both").all()

distinct_day_window_capacity = (
    distinct_day_window_capacity.drop(columns="_merge")
)

assert (
    distinct_day_window_capacity["observed_2plus_windows"]
    == distinct_day_window_capacity[
        "windows_2plus_observed_channels"
    ]
).all()

assert (
    distinct_day_window_capacity["observed_three_windows"]
    == distinct_day_window_capacity["windows_EUH"]
).all()

assert (
    distinct_day_window_capacity["observed_2plus_windows"]
    == (
        distinct_day_window_capacity[
            "distinct_day_2plus_windows"
        ]
        + distinct_day_window_capacity[
            "same_day_only_2plus_windows"
        ]
    )
).all()

assert (
    distinct_day_window_capacity["observed_three_windows"]
    == (
        distinct_day_window_capacity[
            "distinct_day_three_windows"
        ]
        + distinct_day_window_capacity[
            "three_without_assignment_windows"
        ]
    )
).all()

assert (
    distinct_day_window_capacity[
        "distinct_day_three_windows"
    ]
    <= distinct_day_window_capacity[
        "distinct_day_2plus_windows"
    ]
).all()

# One eligible day cannot support different-day assignment.
one_day = distinct_day_window_capacity.loc[
    distinct_day_window_capacity["window_workdays"] == 1
]

assert one_day["distinct_day_2plus_windows"].sum() == 0
assert one_day["distinct_day_three_windows"].sum() == 0

assert len(distinct_day_window_capacity) == len(
    channel_window_capacity
)

# ------------------------------------------------------------
# 6. POPULATION SUMMARY
# ------------------------------------------------------------

distinct_day_window_summary = (
    distinct_day_window_capacity
    .groupby("window_workdays")
    .agg(
        users_with_windows=("user", "size"),
        users_with_observed_2plus=(
            "observed_2plus_windows",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_distinct_day_2plus=(
            "distinct_day_2plus_windows",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_observed_three=(
            "observed_three_windows",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_distinct_day_three=(
            "distinct_day_three_windows",
            lambda x: int(x.gt(0).sum())
        ),
        total_observed_2plus_windows=(
            "observed_2plus_windows", "sum"
        ),
        total_distinct_day_2plus_windows=(
            "distinct_day_2plus_windows", "sum"
        ),
        total_observed_three_windows=(
            "observed_three_windows", "sum"
        ),
        total_distinct_day_three_windows=(
            "distinct_day_three_windows", "sum"
        ),
        total_three_pair_constraint_fail=(
            "three_pair_constraint_fail_windows", "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 78)
print("DISTINCT-DAY CHANNEL SUPPORT — FUTURE PERIOD")
print("=" * 78)

print(
    "User-length records:",
    f"{len(distinct_day_window_capacity):,}"
)

print(
    "Total observed 2+ channel windows:",
    f"{distinct_day_window_capacity['observed_2plus_windows'].sum():,}"
)

print(
    "2+ channels on distinct days:",
    f"{distinct_day_window_capacity['distinct_day_2plus_windows'].sum():,}"
)

print(
    "2+ channels confined to one active day:",
    f"{distinct_day_window_capacity['same_day_only_2plus_windows'].sum():,}"
)

print(
    "Total observed 3-channel windows:",
    f"{distinct_day_window_capacity['observed_three_windows'].sum():,}"
)

print(
    "3 channels assignable to distinct days:",
    f"{distinct_day_window_capacity['distinct_day_three_windows'].sum():,}"
)

print(
    "3-channel windows without distinct-day assignment:",
    f"{distinct_day_window_capacity['three_without_assignment_windows'].sum():,}"
)

print(
    "Of these, failures despite 3+ active days:",
    f"{distinct_day_window_capacity['three_pair_constraint_fail_windows'].sum():,}"
)

# Reporting checkpoints only.
checkpoints = [
    x for x in [1, 2, 3, 5, 10, 20, 40, 60, 79, 80, 100, 108]
    if x <= int(future_daily.groupby("user").size().max())
]

selected = distinct_day_window_summary.loc[
    distinct_day_window_summary[
        "window_workdays"
    ].isin(checkpoints),
    [
        "window_workdays",
        "users_with_observed_2plus",
        "users_with_distinct_day_2plus",
        "users_with_observed_three",
        "users_with_distinct_day_three",
        "total_distinct_day_2plus_windows",
        "total_distinct_day_three_windows"
    ]
]

print("\nPOPULATION COVERAGE — EXPLORATORY WINDOW LENGTHS")
print("-" * 78)
print(selected.to_string(index=False))

print("\nVALIDATION: PASS")
print("No switching order or scenario imposed.")
print("No files saved.")


DISTINCT-DAY CHANNEL SUPPORT — FUTURE PERIOD
User-length records: 387,162
Total observed 2+ channel windows: 5,122,322
2+ channels on distinct days: 5,083,981
2+ channels confined to one active day: 38,341
Total observed 3-channel windows: 1,352,220
3 channels assignable to distinct days: 1,338,581
3-channel windows without distinct-day assignment: 13,639
Of these, failures despite 3+ active days: 4,593

POPULATION COVERAGE — EXPLORATORY WINDOW LENGTHS
------------------------------------------------------------------------------
 window_workdays  users_with_observed_2plus  users_with_distinct_day_2plus  users_with_observed_three  users_with_distinct_day_three  total_distinct_day_2plus_windows  total_distinct_day_three_windows
               1                        985                              0                        239                              0                                 0                                 0
               2                        995                   


### 3.4 Observed Channel Ordering

**Objective:** Identify which channel-order patterns are supported by naturally observed activity on distinct eligible workdays within each candidate window.

**Analysis:**
- Examine all six possible ordered two-channel patterns.
- Examine all twelve three-step patterns where consecutive channels differ.
- Distinguish three-distinct-channel patterns (e.g., E → U → H) from return patterns (e.g., E → U → E).
- Count supporting windows and users for every pattern and window length.
- Validate ordered-channel support against the distinct-day analysis.

**Definitions:**
- A pattern is supported when its channel activities can be matched to different days in chronological order.
- Intermediate days may contain other activity or no target-channel activity.
- A channel may coexist with other channels on its matching day.
- Pattern support does not require direct single-channel switching or imply injection feasibility.

**Methodology:**
- Exhaustively evaluate short ordering motifs rather than choosing a preferred scenario.
- Use the earliest chronological completion of each motif to count supporting windows efficiently.
- Aggregate results by user and window length.
- Preserve longer scenario-pattern discovery for subsequent stages.

**Important:** No attack pattern, duration, minimum switching count, or eligibility rule is imposed. Results remain in memory.


In [8]:

# ============================================================
# 3.4 OBSERVED CHANNEL ORDERING
# ============================================================

from itertools import permutations, product
import numpy as np
import pandas as pd

CHANNEL_INDEX = {"E": 0, "U": 1, "H": 2}
LETTERS = "EUH"

# Exhaustive observational motifs, not attack scenarios.
pair_patterns = list(permutations(LETTERS, 2))

triple_patterns = [
    p for p in product(LETTERS, repeat=3)
    if p[0] != p[1] and p[1] != p[2]
]

distinct_triples = [
    p for p in triple_patterns if len(set(p)) == 3
]

return_triples = [
    p for p in triple_patterns if p[0] == p[2]
]

assert len(pair_patterns) == 6
assert len(triple_patterns) == 12
assert len(distinct_triples) == 6
assert len(return_triples) == 6

# ------------------------------------------------------------
# 1. EARLIEST COMPLETION FOR EACH START POSITION
# ------------------------------------------------------------

def earliest_completion(flags, pattern):
    """
    For each possible starting position, return the
    earliest exclusive end position supporting the
    ordered pattern on distinct days.

    n+1 means the pattern cannot be completed.
    """

    n = len(flags)
    positions = np.arange(n, dtype=np.int32)

    for channel in pattern:

        active_positions = np.flatnonzero(
            flags[:, CHANNEL_INDEX[channel]]
        )

        if len(active_positions) == 0:
            return np.full(n, n + 1, dtype=np.int32)

        indices = np.searchsorted(
            active_positions, positions
        )

        matches = active_positions[
            np.minimum(indices, len(active_positions) - 1)
        ]

        positions = np.where(
            indices < len(active_positions),
            matches + 1,
            n + 1
        )

    return positions


def count_supported_windows(completion):
    """
    Count supporting windows at every possible length.
    One window is counted once per pattern, even if
    multiple matching subsequences exist within it.
    """

    n = len(completion)
    starts = np.arange(n, dtype=np.int32)

    valid = completion <= n

    minimum_length = (
        completion[valid] - starts[valid]
    )

    maximum_length = (
        n - starts[valid]
    )

    difference = np.zeros(n + 2, dtype=np.int64)

    np.add.at(difference, minimum_length, 1)
    np.add.at(difference, maximum_length + 1, -1)

    return np.cumsum(difference)[1:n + 1]


# ------------------------------------------------------------
# 2. EVALUATE ALL USERS
# ------------------------------------------------------------

ordering_parts = []

for user, group in future_daily.groupby("user", sort=False):

    flags = group[
        ["email_active", "usb_active", "http_active"]
    ].to_numpy(dtype=np.int8)

    n = len(flags)

    if n == 0:
        continue

    all_patterns = pair_patterns + triple_patterns

    completion_map = {
        p: earliest_completion(flags, p)
        for p in all_patterns
    }

    part = pd.DataFrame({
        "user": [user] * n,
        "window_workdays": np.arange(1, n + 1)
    })

    # Counts for each individual ordering motif.
    for pattern, completion in completion_map.items():
        label = "_".join(pattern)
        part[f"ordered_{label}"] = (
            count_supported_windows(completion)
        )

    # Union: at least one matching motif of each type.
    motif_groups = {
        "windows_any_ordered_pair": pair_patterns,
        "windows_any_three_step_change": triple_patterns,
        "windows_three_distinct_order": distinct_triples,
        "windows_return_pattern": return_triples
    }

    for column, patterns in motif_groups.items():

        earliest = np.minimum.reduce([
            completion_map[p] for p in patterns
        ])

        part[column] = count_supported_windows(earliest)

    part["supported_pair_pattern_types"] = sum(
        part[f"ordered_{'_'.join(p)}"].gt(0)
        for p in pair_patterns
    ).astype(int)

    part["supported_three_step_pattern_types"] = sum(
        part[f"ordered_{'_'.join(p)}"].gt(0)
        for p in triple_patterns
    ).astype(int)

    ordering_parts.append(part)

ordered_window_capacity = pd.concat(
    ordering_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. CROSS-VALIDATE AGAINST SECTION 3.3
# ------------------------------------------------------------

reference = distinct_day_window_capacity[
    [
        "user",
        "window_workdays",
        "distinct_day_2plus_windows",
        "distinct_day_three_windows"
    ]
]

ordered_window_capacity = ordered_window_capacity.merge(
    reference,
    on=["user", "window_workdays"],
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert ordered_window_capacity["_merge"].eq("both").all()
ordered_window_capacity.drop(columns="_merge", inplace=True)

# Any two different channels on distinct days
# must support at least one chronological ordering.
assert (
    ordered_window_capacity["windows_any_ordered_pair"]
    == ordered_window_capacity["distinct_day_2plus_windows"]
).all(), "Two-channel ordering validation failed"

# Any three channels assignable to distinct days
# must support at least one of the six orderings.
assert (
    ordered_window_capacity["windows_three_distinct_order"]
    == ordered_window_capacity["distinct_day_three_windows"]
).all(), "Three-channel ordering validation failed"

assert (
    ordered_window_capacity["windows_three_distinct_order"]
    <= ordered_window_capacity["windows_any_three_step_change"]
).all()

assert (
    ordered_window_capacity["windows_return_pattern"]
    <= ordered_window_capacity["windows_any_three_step_change"]
).all()

assert (
    ordered_window_capacity["supported_pair_pattern_types"]
    .between(0, 6)
).all()

assert (
    ordered_window_capacity["supported_three_step_pattern_types"]
    .between(0, 12)
).all()

one_day = ordered_window_capacity.loc[
    ordered_window_capacity["window_workdays"] == 1
]

assert one_day["windows_any_ordered_pair"].sum() == 0
assert one_day["windows_any_three_step_change"].sum() == 0

assert len(ordered_window_capacity) == len(
    distinct_day_window_capacity
)

# ------------------------------------------------------------
# 4. POPULATION SUMMARY
# ------------------------------------------------------------

ordered_window_summary = (
    ordered_window_capacity
    .groupby("window_workdays")
    .agg(
        users_with_windows=("user", "size"),
        users_with_ordered_pair=(
            "windows_any_ordered_pair",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_three_step_change=(
            "windows_any_three_step_change",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_three_distinct_order=(
            "windows_three_distinct_order",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_return_pattern=(
            "windows_return_pattern",
            lambda x: int(x.gt(0).sum())
        ),
        total_windows_with_three_step_change=(
            "windows_any_three_step_change", "sum"
        ),
        total_windows_with_return_pattern=(
            "windows_return_pattern", "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 76)
print("OBSERVED CHANNEL ORDERING — FUTURE PERIOD")
print("=" * 76)

print(f"Two-step motif types   : {len(pair_patterns)}")
print(f"Three-step motif types : {len(triple_patterns)}")
print(f"User-length records    : {len(ordered_window_capacity):,}")

checkpoints = [
    x for x in [1, 2, 3, 5, 10, 20, 40, 60, 80, 100, 108]
    if x <= int(future_daily.groupby("user").size().max())
]

display_cols = [
    "window_workdays",
    "users_with_windows",
    "users_with_ordered_pair",
    "users_with_three_step_change",
    "users_with_three_distinct_order",
    "users_with_return_pattern"
]

print("\nUSER COVERAGE — EXPLORATORY WINDOW LENGTHS")
print("-" * 76)

print(
    ordered_window_summary.loc[
        ordered_window_summary["window_workdays"].isin(checkpoints),
        display_cols
    ].to_string(index=False)
)

# Pattern-specific coverage at an illustrative length.
# This does not rank or select attack scenarios.
example_length = 20

example = ordered_window_capacity.loc[
    ordered_window_capacity["window_workdays"] == example_length
]

pattern_coverage = pd.DataFrame([
    {
        "observed_pattern": " -> ".join(p),
        "users": int(
            example[f"ordered_{'_'.join(p)}"].gt(0).sum()
        ),
        "supporting_windows": int(
            example[f"ordered_{'_'.join(p)}"].sum()
        )
    }
    for p in all_patterns
]).sort_values(
    ["users", "observed_pattern"],
    ascending=[False, True]
)

print(f"\nOBSERVED ORDERING SUPPORT — {example_length} WORKDAYS")
print("-" * 76)
print(pattern_coverage.to_string(index=False))

print("\nVALIDATION: PASS")
print("All ordering motifs evaluated without scenario assignment.")
print("No files saved.")


OBSERVED CHANNEL ORDERING — FUTURE PERIOD
Two-step motif types   : 6
Three-step motif types : 12
User-length records    : 387,162

USER COVERAGE — EXPLORATORY WINDOW LENGTHS
----------------------------------------------------------------------------
 window_workdays  users_with_windows  users_with_ordered_pair  users_with_three_step_change  users_with_three_distinct_order  users_with_return_pattern
               1                3737                        0                             0                                0                          0
               2                3736                      994                             0                                0                          0
               3                3736                      994                           947                              273                        946
               5                3736                      995                           989                              288                 


### 3.5 Exclusive vs. Overlapping Channel-Order Support

**Objective:** Compare two interpretations of naturally observed channel-order patterns within candidate windows.

**Flexible Observed Support**
- A channel is considered present if it is active on a matching day.
- Other target channels may also be active on that day.
- This is the definition used in Section 3.4.

**Exclusive Observed Support**
- A matching day must contain activity in only the selected target channel.
- E requires an E-only day, U requires a U-only day, and H requires an H-only day.
- Pattern steps must occur on different days in chronological order.
- Other non-target activities are not restricted.

**Analysis:**
- Reevaluate all 6 two-step and 12 three-step patterns.
- Compare flexible and exclusive support for each window length.
- Quantify how many users and windows retain ordered-pattern support.
- Distinguish missing exclusive evidence from missing channel availability.

**Interpretation:**
Exclusive support is a stricter description of natural behavior, not a mandatory attack condition.

Neither definition establishes injection feasibility, stealth, attack duration, or final scenario eligibility.

**Outputs:** In-memory comparative capacity and summary tables. No files are saved.


In [9]:

# ============================================================
# 3.5 EXCLUSIVE VS OVERLAPPING CHANNEL-ORDER SUPPORT
# ============================================================

import numpy as np
import pandas as pd
from itertools import permutations, product

letters = "EUH"

pair_patterns = list(permutations(letters, 2))

triple_patterns = [
    p for p in product(letters, repeat=3)
    if p[0] != p[1] and p[1] != p[2]
]

all_patterns = pair_patterns + triple_patterns

distinct_triples = [
    p for p in triple_patterns if len(set(p)) == 3
]

return_triples = [
    p for p in triple_patterns if p[0] == p[2]
]

assert len(all_patterns) == 18

# Reuse the validated functions from Section 3.4.
assert callable(earliest_completion)
assert callable(count_supported_windows)

# ------------------------------------------------------------
# 1. EVALUATE EXCLUSIVE CHANNEL STATES
# ------------------------------------------------------------

exclusive_parts = []

for user, group in future_daily.groupby("user", sort=False):

    flags = group[
        ["email_active", "usb_active", "http_active"]
    ].to_numpy(dtype=np.int8)

    n = len(flags)

    if n == 0:
        continue

    # E=1, U=2, H=4; combinations have other codes.
    daily_state = flags @ np.array([1, 2, 4])

    # Exactly one target channel must be active.
    exclusive_flags = np.column_stack([
        daily_state == 1,
        daily_state == 2,
        daily_state == 4
    ]).astype(np.int8)

    completion_map = {
        p: earliest_completion(exclusive_flags, p)
        for p in all_patterns
    }

    part = pd.DataFrame({
        "user": [user] * n,
        "window_workdays": np.arange(1, n + 1)
    })

    for pattern, completion in completion_map.items():
        name = "_".join(pattern)

        part[f"exclusive_{name}"] = (
            count_supported_windows(completion)
        )

    groups = {
        "exclusive_any_ordered_pair": pair_patterns,
        "exclusive_any_three_step_change": triple_patterns,
        "exclusive_three_distinct_order": distinct_triples,
        "exclusive_return_pattern": return_triples
    }

    for column, patterns in groups.items():

        earliest = np.minimum.reduce([
            completion_map[p] for p in patterns
        ])

        part[column] = count_supported_windows(earliest)

    # --------------------------------------------------------
    # 2. INDEPENDENT EXACT-DAY CROSS-CHECK
    # --------------------------------------------------------

    prefix = np.vstack([
        np.zeros((1, 3), dtype=np.int32),
        np.cumsum(exclusive_flags, axis=0, dtype=np.int32)
    ])

    starts, ends = np.triu_indices(n)

    exact_day_counts = (
        prefix[ends + 1] - prefix[starts]
    )

    lengths = ends - starts + 1

    expected_pair = np.bincount(
        lengths[
            (exact_day_counts > 0).sum(axis=1) >= 2
        ] - 1,
        minlength=n
    )

    expected_three = np.bincount(
        lengths[
            (exact_day_counts > 0).all(axis=1)
        ] - 1,
        minlength=n
    )

    assert np.array_equal(
        part["exclusive_any_ordered_pair"].to_numpy(),
        expected_pair
    ), f"Exclusive pair mismatch: {user}"

    assert np.array_equal(
        part["exclusive_three_distinct_order"].to_numpy(),
        expected_three
    ), f"Exclusive triple mismatch: {user}"

    exclusive_parts.append(part)

exclusive_order_capacity = pd.concat(
    exclusive_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 3. COMPARE WITH FLEXIBLE OBSERVED SUPPORT
# ------------------------------------------------------------

flexible_columns = [
    "user",
    "window_workdays",
    "windows_any_ordered_pair",
    "windows_any_three_step_change",
    "windows_three_distinct_order",
    "windows_return_pattern"
]

exclusive_order_capacity = (
    exclusive_order_capacity.merge(
        ordered_window_capacity[flexible_columns],
        on=["user", "window_workdays"],
        how="outer",
        validate="one_to_one",
        indicator=True
    )
)

assert exclusive_order_capacity["_merge"].eq("both").all()

exclusive_order_capacity.drop(
    columns="_merge",
    inplace=True
)

comparisons = {
    "exclusive_any_ordered_pair":
        "windows_any_ordered_pair",
    "exclusive_any_three_step_change":
        "windows_any_three_step_change",
    "exclusive_three_distinct_order":
        "windows_three_distinct_order",
    "exclusive_return_pattern":
        "windows_return_pattern"
}

for strict_col, flexible_col in comparisons.items():

    assert (
        exclusive_order_capacity[strict_col]
        <= exclusive_order_capacity[flexible_col]
    ).all(), f"Exclusive support exceeds flexible: {strict_col}"

# ------------------------------------------------------------
# 4. POPULATION SUMMARY
# ------------------------------------------------------------

exclusive_order_summary = (
    exclusive_order_capacity
    .groupby("window_workdays")
    .agg(
        users_with_flexible_pair=(
            "windows_any_ordered_pair",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_exclusive_pair=(
            "exclusive_any_ordered_pair",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_flexible_three_step=(
            "windows_any_three_step_change",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_exclusive_three_step=(
            "exclusive_any_three_step_change",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_flexible_three_channels=(
            "windows_three_distinct_order",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_exclusive_three_channels=(
            "exclusive_three_distinct_order",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_flexible_return=(
            "windows_return_pattern",
            lambda x: int(x.gt(0).sum())
        ),
        users_with_exclusive_return=(
            "exclusive_return_pattern",
            lambda x: int(x.gt(0).sum())
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

assert len(exclusive_order_capacity) == len(
    ordered_window_capacity
)

assert not exclusive_order_capacity.duplicated(
    ["user", "window_workdays"]
).any()

one_day = exclusive_order_capacity.loc[
    exclusive_order_capacity["window_workdays"] == 1
]

assert (
    one_day["exclusive_any_ordered_pair"].sum() == 0
)

assert (
    one_day["exclusive_any_three_step_change"].sum() == 0
)

# ------------------------------------------------------------
# 6. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 76)
print("EXCLUSIVE VS FLEXIBLE CHANNEL ORDERING")
print("=" * 76)

print(
    "User-length records:",
    f"{len(exclusive_order_capacity):,}"
)

checkpoints = [
    x for x in [2, 3, 5, 10, 20, 40, 80, 100]
    if x <= future_daily.groupby("user").size().max()
]

selected = exclusive_order_summary.loc[
    exclusive_order_summary[
        "window_workdays"
    ].isin(checkpoints)
]

print("\nUSER COVERAGE BY WINDOW LENGTH")
print("-" * 76)

print(selected.to_string(index=False))

# ------------------------------------------------------------
# 7. PATTERN-LEVEL COMPARISON AT 20 WORKDAYS
# ------------------------------------------------------------

example_length = 20

strict_example = exclusive_order_capacity.loc[
    exclusive_order_capacity["window_workdays"] == example_length
]

flex_example = ordered_window_capacity.loc[
    ordered_window_capacity["window_workdays"] == example_length
]

pattern_comparison = pd.DataFrame([
    {
        "pattern": " -> ".join(p),
        "flexible_users": int(
            flex_example[
                f"ordered_{'_'.join(p)}"
            ].gt(0).sum()
        ),
        "exclusive_users": int(
            strict_example[
                f"exclusive_{'_'.join(p)}"
            ].gt(0).sum()
        )
    }
    for p in all_patterns
])

pattern_comparison["difference"] = (
    pattern_comparison["flexible_users"]
    - pattern_comparison["exclusive_users"]
)

pattern_comparison = pattern_comparison.sort_values(
    ["difference", "pattern"],
    ascending=[False, True]
)

print("\nORDERING PATTERNS — 20 ELIGIBLE WORKDAYS")
print("-" * 76)
print(pattern_comparison.to_string(index=False))

print("\nVALIDATION: PASS")
print("Exclusive support is descriptive, not an eligibility rule.")
print("No files saved.")


EXCLUSIVE VS FLEXIBLE CHANNEL ORDERING
User-length records: 387,162

USER COVERAGE BY WINDOW LENGTH
----------------------------------------------------------------------------
 window_workdays  users_with_flexible_pair  users_with_exclusive_pair  users_with_flexible_three_step  users_with_exclusive_three_step  users_with_flexible_three_channels  users_with_exclusive_three_channels  users_with_flexible_return  users_with_exclusive_return
               2                       994                        919                               0                                0                                   0                                    0                           0                            0
               3                       994                        930                             947                              737                                 273                                   87                         946                          719
               5            


### 3.6 Window Support Strength and Temporal Flexibility

**Objective:** Measure how consistently naturally observed channel-order patterns are supported across the available windows of each user.

**Analysis:**
- Calculate the percentage of candidate windows supporting each ordering category.
- Compare flexible and exclusive natural support.
- Measure the diversity of supported two-step and three-step motifs.
- Compare temporal flexibility across all supported window lengths.
- Distinguish users with one supporting window from users with many supporting windows.

**Interpretation:**
- A higher support percentage indicates more observed temporal options, not greater attack eligibility.
- Overlapping windows are alternative candidate placements, not independent attacks.
- Support is based on natural activity and does not restrict future injection to those exact days.
- No preferred attack length, minimum support percentage, or switching pattern is imposed.

**Expected Outputs (in memory):**
- `window_support_strength`
- `window_support_summary`

Final results will be consolidated and saved after completing the window audit.


In [10]:

# ============================================================
# 3.6 WINDOW SUPPORT STRENGTH
# ============================================================

import pandas as pd
import numpy as np

KEYS = ["user", "window_workdays"]

# ------------------------------------------------------------
# 1. COMBINE PREVIOUSLY VALIDATED RESULTS
# ------------------------------------------------------------

flexible_columns = [
    "windows_any_ordered_pair",
    "windows_any_three_step_change",
    "windows_three_distinct_order",
    "windows_return_pattern",
    "supported_pair_pattern_types",
    "supported_three_step_pattern_types"
]

exclusive_columns = [
    "exclusive_any_ordered_pair",
    "exclusive_any_three_step_change",
    "exclusive_three_distinct_order",
    "exclusive_return_pattern"
]

window_support_strength = (
    window_length_capacity[
        KEYS + ["eligible_windows"]
    ]
    .merge(
        ordered_window_capacity[
            KEYS + flexible_columns
        ],
        on=KEYS,
        validate="one_to_one"
    )
    .merge(
        exclusive_order_capacity[
            KEYS + exclusive_columns
        ],
        on=KEYS,
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 2. CALCULATE SUPPORT PERCENTAGES
# ------------------------------------------------------------

support_columns = flexible_columns[:4] + exclusive_columns

for column in support_columns:

    pct_column = column + "_pct"

    window_support_strength[pct_column] = (
        100
        * window_support_strength[column]
        / window_support_strength["eligible_windows"]
    ).round(2)

# Distinct motif types supported in at least one window
# of the given length; not the number within one window.
window_support_strength[
    "pair_pattern_type_count"
] = window_support_strength[
    "supported_pair_pattern_types"
]

window_support_strength[
    "three_step_pattern_type_count"
] = window_support_strength[
    "supported_three_step_pattern_types"
]

# ------------------------------------------------------------
# 3. VALIDATION
# ------------------------------------------------------------

assert len(window_support_strength) == len(
    window_length_capacity
)

assert not window_support_strength.duplicated(
    KEYS
).any()

assert window_support_strength[
    "eligible_windows"
].gt(0).all()

for column in support_columns:

    assert (
        window_support_strength[column]
        <= window_support_strength["eligible_windows"]
    ).all()

    assert window_support_strength[
        column + "_pct"
    ].between(0, 100).all()

for flexible, exclusive in [
    ("windows_any_ordered_pair", "exclusive_any_ordered_pair"),
    ("windows_any_three_step_change", "exclusive_any_three_step_change"),
    ("windows_three_distinct_order", "exclusive_three_distinct_order"),
    ("windows_return_pattern", "exclusive_return_pattern")
]:
    assert (
        window_support_strength[exclusive]
        <= window_support_strength[flexible]
    ).all()

assert window_support_strength[
    "pair_pattern_type_count"
].between(0, 6).all()

assert window_support_strength[
    "three_step_pattern_type_count"
].between(0, 12).all()

assert (
    window_support_strength["eligible_windows"].sum()
    == 20_436_421
)

# ------------------------------------------------------------
# 4. SUMMARIZE BY WINDOW LENGTH
# ------------------------------------------------------------

def positive_median(series):
    positive = series[series > 0]
    return (
        round(float(positive.median()), 2)
        if len(positive) else np.nan
    )

window_support_summary = (
    window_support_strength
    .groupby("window_workdays")
    .agg(
        users_with_windows=("user", "size"),
        total_windows=("eligible_windows", "sum"),
        users_with_flexible_pair=(
            "windows_any_ordered_pair",
            lambda s: int(s.gt(0).sum())
        ),
        users_with_flexible_three_step=(
            "windows_any_three_step_change",
            lambda s: int(s.gt(0).sum())
        ),
        users_with_three_distinct=(
            "windows_three_distinct_order",
            lambda s: int(s.gt(0).sum())
        ),
        median_pair_window_support_pct=(
            "windows_any_ordered_pair_pct",
            positive_median
        ),
        median_three_step_window_support_pct=(
            "windows_any_three_step_change_pct",
            positive_median
        ),
        median_three_distinct_window_support_pct=(
            "windows_three_distinct_order_pct",
            positive_median
        ),
        median_pair_pattern_types=(
            "pair_pattern_type_count",
            positive_median
        ),
        median_three_step_pattern_types=(
            "three_step_pattern_type_count",
            positive_median
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. DISPLAY CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 78)
print("WINDOW SUPPORT STRENGTH — FUTURE PERIOD")
print("=" * 78)

print(
    "User-length records:",
    f"{len(window_support_strength):,}"
)

print(
    "Total candidate windows:",
    f"{window_support_strength['eligible_windows'].sum():,}"
)

checkpoints = [
    x for x in [2, 3, 5, 10, 20, 40, 60, 80, 100]
    if x <= window_support_summary["window_workdays"].max()
]

selected_columns = [
    "window_workdays",
    "users_with_windows",
    "users_with_flexible_pair",
    "users_with_flexible_three_step",
    "users_with_three_distinct",
    "median_pair_window_support_pct",
    "median_three_step_window_support_pct",
    "median_three_distinct_window_support_pct"
]

print("\nTEMPORAL SUPPORT COVERAGE")
print("-" * 78)

print(
    window_support_summary.loc[
        window_support_summary["window_workdays"].isin(
            checkpoints
        ),
        selected_columns
    ].to_string(index=False)
)

print("\nPATTERN DIVERSITY BY WINDOW LENGTH")
print("-" * 78)

print(
    window_support_summary.loc[
        window_support_summary["window_workdays"].isin(
            checkpoints
        ),
        [
            "window_workdays",
            "median_pair_pattern_types",
            "median_three_step_pattern_types"
        ]
    ].to_string(index=False)
)

print("\nVALIDATION: PASS")
print("No scenario constraints imposed.")
print("No files saved.")


WINDOW SUPPORT STRENGTH — FUTURE PERIOD
User-length records: 387,162
Total candidate windows: 20,436,421

TEMPORAL SUPPORT COVERAGE
------------------------------------------------------------------------------
 window_workdays  users_with_windows  users_with_flexible_pair  users_with_flexible_three_step  users_with_three_distinct  median_pair_window_support_pct  median_three_step_window_support_pct  median_three_distinct_window_support_pct
               2                3736                       994                               0                          0                           34.29                                   NaN                                       NaN
               3                3736                       994                             947                        273                           56.31                                 18.45                                     12.62
               5                3736                       995                         


### 3.7 Channel Recurrence Within Candidate Windows

**Objective:** Measure how frequently each channel appears within the available windows, beyond simple channel presence or ordering support.

**Analysis:**
- Count naturally active days per channel within each window.
- Identify windows containing two or three distinct channels.
- Identify windows where at least two channels each appear on 2 or more days.
- Identify windows where all three channels each appear on 2 or more days.
- Repeat the comparison using 3 or more active days per channel.
- Measure population coverage across all supported window lengths.

**Interpretation:**
- These activity counts describe the depth of natural channel support.
- Repeated channel activity does not necessarily imply exclusive or consecutive activity.
- Thresholds of 2 and 3 days are descriptive comparisons, not attack requirements.
- A channel can still be selected for injection on an eligible day without observed natural activity in that channel, subject to later simulation rules.
- No scenario selection or injection occurs.

**Expected Results:**
- `channel_recurrence_capacity`
- `channel_recurrence_summary`

Results remain in memory until the final Notebook 02 export.


In [11]:

# ============================================================
# 3.7 CHANNEL RECURRENCE WITHIN CANDIDATE WINDOWS
# ============================================================

import numpy as np
import pandas as pd

KEYS = ["user", "window_workdays"]
FLAGS = ["email_active", "usb_active", "http_active"]

recurrence_parts = []

# ------------------------------------------------------------
# 1. CALCULATE CHANNEL-ACTIVE DAYS IN EACH WINDOW
# ------------------------------------------------------------

for user, group in future_daily.groupby("user", sort=False):

    flags = group[FLAGS].to_numpy(dtype=np.int32)
    n = len(flags)

    if n == 0:
        continue

    prefix = np.vstack([
        np.zeros((1, 3), dtype=np.int32),
        np.cumsum(flags, axis=0, dtype=np.int32)
    ])

    starts, ends = np.triu_indices(n)
    lengths = ends - starts + 1

    # Number of active days per channel in each window.
    counts = prefix[ends + 1] - prefix[starts]

    observed_channels = (counts >= 1).sum(axis=1)
    channels_with_2days = (counts >= 2).sum(axis=1)
    channels_with_3days = (counts >= 3).sum(axis=1)

    def window_count(mask):
        return np.bincount(
            lengths[mask] - 1,
            minlength=n
        ).astype(np.int64)

    part = pd.DataFrame({
        "user": [user] * n,
        "window_workdays": np.arange(1, n + 1),

        "windows_2plus_observed":
            window_count(observed_channels >= 2),

        "windows_3_observed":
            window_count(observed_channels == 3),

        "windows_2plus_channels_each_2days":
            window_count(channels_with_2days >= 2),

        "windows_3_channels_each_2days":
            window_count(channels_with_2days == 3),

        "windows_2plus_channels_each_3days":
            window_count(channels_with_3days >= 2),

        "windows_3_channels_each_3days":
            window_count(channels_with_3days == 3)
    })

    # Sum of active-day counts across all overlapping windows.
    # Useful for calculating average channel recurrence later.
    for i, channel in enumerate(["email", "usb", "http"]):
        part[f"{channel}_active_day_occurrences"] = (
            np.bincount(
                lengths - 1,
                weights=counts[:, i],
                minlength=n
            ).astype(np.int64)
        )

    recurrence_parts.append(part)

channel_recurrence_capacity = pd.concat(
    recurrence_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 2. VALIDATE AGAINST PREVIOUS WINDOW ANALYSIS
# ------------------------------------------------------------

reference = distinct_day_window_capacity[
    KEYS + [
        "windows_2plus_observed_channels",
        "windows_EUH",
        "distinct_day_2plus_windows",
        "distinct_day_three_windows"
    ]
]

channel_recurrence_capacity = (
    channel_recurrence_capacity.merge(
        reference,
        on=KEYS,
        validate="one_to_one",
        how="inner"
    )
)

assert len(channel_recurrence_capacity) == len(
    window_length_capacity
)

assert not channel_recurrence_capacity.duplicated(
    KEYS
).any()

assert (
    channel_recurrence_capacity["windows_2plus_observed"]
    == channel_recurrence_capacity[
        "windows_2plus_observed_channels"
    ]
).all()

assert (
    channel_recurrence_capacity["windows_3_observed"]
    == channel_recurrence_capacity["windows_EUH"]
).all()

# Higher recurrence must be a subset of lower recurrence.
for prefix in ["windows_2plus_channels", "windows_3_channels"]:

    assert (
        channel_recurrence_capacity[f"{prefix}_each_3days"]
        <= channel_recurrence_capacity[f"{prefix}_each_2days"]
    ).all()

assert (
    channel_recurrence_capacity[
        "windows_2plus_channels_each_2days"
    ]
    <= channel_recurrence_capacity[
        "distinct_day_2plus_windows"
    ]
).all()

assert (
    channel_recurrence_capacity[
        "windows_3_channels_each_3days"
    ]
    <= channel_recurrence_capacity[
        "distinct_day_three_windows"
    ]
).all()

# One-day windows cannot contain two active days per channel.
one_day = channel_recurrence_capacity.loc[
    channel_recurrence_capacity["window_workdays"] == 1
]

assert one_day[
    "windows_2plus_channels_each_2days"
].sum() == 0

assert one_day[
    "windows_3_channels_each_2days"
].sum() == 0

# ------------------------------------------------------------
# 3. POPULATION SUMMARY
# ------------------------------------------------------------

metric_columns = [
    "windows_2plus_observed",
    "windows_3_observed",
    "windows_2plus_channels_each_2days",
    "windows_3_channels_each_2days",
    "windows_2plus_channels_each_3days",
    "windows_3_channels_each_3days"
]

summary_parts = []

for length, group in channel_recurrence_capacity.groupby(
    "window_workdays"
):

    record = {
        "window_workdays": length,
        "users_with_windows": len(group)
    }

    for column in metric_columns:
        record[f"users_{column}"] = int(
            group[column].gt(0).sum()
        )
        record[f"total_{column}"] = int(
            group[column].sum()
        )

    summary_parts.append(record)

channel_recurrence_summary = pd.DataFrame(summary_parts)

# ------------------------------------------------------------
# 4. CONSOLIDATED RESULTS
# ------------------------------------------------------------

print("=" * 78)
print("CHANNEL RECURRENCE WITHIN CANDIDATE WINDOWS")
print("=" * 78)

print(
    "User-length records:",
    f"{len(channel_recurrence_capacity):,}"
)

checkpoints = [
    length for length in
    [1, 2, 3, 5, 10, 20, 40, 60, 80, 100]
    if length <= int(channel_recurrence_summary["window_workdays"].max())
]

selected = channel_recurrence_summary.loc[
    channel_recurrence_summary["window_workdays"].isin(
        checkpoints
    ),
    [
        "window_workdays",
        "users_with_windows",
        "users_windows_2plus_observed",
        "users_windows_2plus_channels_each_2days",
        "users_windows_2plus_channels_each_3days",
        "users_windows_3_observed",
        "users_windows_3_channels_each_2days",
        "users_windows_3_channels_each_3days"
    ]
]

print("\nUSER COVERAGE — CHANNEL RECURRENCE")
print("-" * 78)
print(selected.to_string(index=False))

print("\nWINDOW COUNTS — 20 ELIGIBLE WORKDAYS")
print("-" * 78)

example = channel_recurrence_summary.loc[
    channel_recurrence_summary["window_workdays"] == 20
]

if not example.empty:
    for metric in metric_columns:
        print(
            f"{metric:<42}",
            f"{int(example[f'total_{metric}'].iloc[0]):,}"
        )

print("\nVALIDATION: PASS")
print("Recurrence thresholds are descriptive only.")
print("No scenario or switching sequence imposed.")
print("No files saved.")


CHANNEL RECURRENCE WITHIN CANDIDATE WINDOWS
User-length records: 387,162

USER COVERAGE — CHANNEL RECURRENCE
------------------------------------------------------------------------------
 window_workdays  users_with_windows  users_windows_2plus_observed  users_windows_2plus_channels_each_2days  users_windows_2plus_channels_each_3days  users_windows_3_observed  users_windows_3_channels_each_2days  users_windows_3_channels_each_3days
               1                3737                           985                                        0                                        0                       239                                    0                                    0
               2                3736                           995                                      764                                        0                       280                                   60                                    0
               3                3736                           99


### 3.8 Long-Sequence Channel Rotation Feasibility

**Objective:** Determine how long a channel-rotation sequence can be supported by each user's naturally active channel-days within candidate windows.

**Eligibility Rules:**
- Analyze only users with at least two naturally observed Future channels.
- Each selected step must use a channel active on that exact user-day.
- Each step must occur on a different eligible workday.
- Consecutive steps must use different channels.
- Inactive or unselected days can be skipped.
- At most one channel is assigned to each day in the analytical sequence.

**Analysis:**
- Examine all supported eligible-workday window lengths.
- Discover the maximum alternating sequence length without predefining channel order.
- Count windows supporting longer sequences of 4, 5, 6, 8, 10, 15, 20, and 30 steps.
- Separately evaluate sequences that use all three distinct channels.
- Compare results with the previously validated two-step and three-step analyses.
- Measure user coverage and the number of temporal alternatives at different window lengths.

**Definitions:**
- **Rotation step:** One selected day-channel assignment.
- **Channel switch:** A change between two consecutive selected steps.
- **Maximum feasible steps:** The longest alternating sequence supported by observed channel-day activity in a window.
- **Three-channel sequence:** An alternating sequence containing Email, USB, and HTTP at least once each.
- **Off-day:** A day without attack injection; natural activity may still exist.

**Interpretation:**
These are upper bounds on natural channel-day sequence support. They are not final attack schedules and do not establish volume or stealth feasibility.

No predefined attack sequence, minimum number of switches, or mandatory attack duration is imposed.

Results remain in memory.


In [12]:

# ============================================================
# 3.8 LONG-SEQUENCE CHANNEL ROTATION FEASIBILITY
# ============================================================

import numpy as np
import pandas as pd

KEYS = ["user", "window_workdays"]
FLAGS = ["email_active", "usb_active", "http_active"]

# Observational checkpoints, NOT attack requirements.
STEP_THRESHOLDS = [2, 3, 4, 5, 6, 8, 10, 15, 20, 30]

# ------------------------------------------------------------
# 1. SELECT NATURALLY MULTI-CHANNEL USERS
# ------------------------------------------------------------

candidate_users = set(
    user_profile.loc[
        user_profile["active_channel_count"] >= 2,
        "user"
    ]
)

assert len(candidate_users) == 995

candidate_daily = future_daily.loc[
    future_daily["user"].isin(candidate_users)
].copy()

candidate_daily = candidate_daily.sort_values(
    ["user", "day"]
)

# ------------------------------------------------------------
# 2. DYNAMIC PROGRAMMING — ALL WINDOW POSITIONS
# ------------------------------------------------------------

rotation_parts = []

for user, group in candidate_daily.groupby(
    "user", sort=False
):

    flags = group[FLAGS].to_numpy(dtype=np.int8)
    n = len(flags)

    if n == 0:
        continue

    # DP state:
    # [window_start, last_channel, used_channel_mask]
    #
    # Mask bits: E=1, U=2, H=4.
    # A mask of 7 means all three channels were used.
    #
    # -1 means no valid sequence for this state.
    # Only previous-day DP states can extend a sequence,
    # preventing multiple steps on the same day.

    dp = np.full(
        (n, 3, 8),
        -1,
        dtype=np.int16
    )

    max_any = np.zeros(n, dtype=np.int16)
    max_three = np.zeros(n, dtype=np.int16)

    sum_any = np.zeros(n, dtype=np.int64)

    support_any = {
        k: np.zeros(n, dtype=np.int64)
        for k in STEP_THRESHOLDS
    }

    support_three = {
        k: np.zeros(n, dtype=np.int64)
        for k in STEP_THRESHOLDS
        if k >= 3
    }

    # End position of one window attaining the
    # maximum alternating length at each window size.
    example_end = np.full(n, -1, dtype=np.int32)

    for end in range(n):

        previous = dp[:end + 1].copy()
        updated = previous.copy()

        active_channels = np.flatnonzero(flags[end])

        for channel in active_channels:

            bit = 1 << int(channel)

            # Start a new one-step sequence today.
            updated[:, channel, bit] = np.maximum(
                updated[:, channel, bit],
                1
            )

            # Extend only sequences whose LAST channel
            # differs from today's selected channel.
            other1 = (int(channel) + 1) % 3
            other2 = (int(channel) + 2) % 3

            previous_other = np.maximum(
                previous[:, other1, :],
                previous[:, other2, :]
            )

            for old_mask in range(1, 8):

                new_mask = old_mask | bit

                old_length = previous_other[:, old_mask]

                candidate_length = np.where(
                    old_length >= 1,
                    old_length + 1,
                    -1
                )

                updated[:, channel, new_mask] = np.maximum(
                    updated[:, channel, new_mask],
                    candidate_length
                )

        dp[:end + 1] = updated

        # Longest supported sequence for each window
        # ending at this position.
        best_any = np.maximum(
            updated.max(axis=(1, 2)),
            0
        )[::-1]

        # Longest sequence that uses ALL three channels.
        best_three = np.maximum(
            updated[:, :, 7].max(axis=1),
            0
        )[::-1]

        length_count = end + 1

        previous_best = max_any[:length_count].copy()

        improved = best_any > previous_best

        max_any[:length_count] = np.maximum(
            previous_best,
            best_any
        )

        max_three[:length_count] = np.maximum(
            max_three[:length_count],
            best_three
        )

        sum_any[:length_count] += best_any

        # Store a representative window position.
        example_end[np.flatnonzero(improved)] = end

        for k in STEP_THRESHOLDS:
            support_any[k][:length_count] += (
                best_any >= k
            )

            if k >= 3:
                support_three[k][:length_count] += (
                    best_three >= k
                )

    # --------------------------------------------------------
    # 3. USER × WINDOW-LENGTH RESULTS
    # --------------------------------------------------------

    lengths = np.arange(1, n + 1, dtype=np.int32)
    window_counts = n - lengths + 1

    part = pd.DataFrame({
        "user": user,
        "window_workdays": lengths,
        "eligible_windows": window_counts,
        "max_feasible_rotation_steps": max_any,
        "max_feasible_switches": np.maximum(
            max_any - 1, 0
        ),
        "max_steps_using_3channels": max_three,
        "mean_max_steps_across_windows": np.round(
            sum_any / window_counts, 2
        ),
        "example_window_end_position": example_end
    })

    for k in STEP_THRESHOLDS:

        part[f"windows_supporting_{k}steps"] = (
            support_any[k]
        )

        if k >= 3:
            part[f"windows_3channel_{k}steps"] = (
                support_three[k]
            )

    rotation_parts.append(part)

long_rotation_capacity = pd.concat(
    rotation_parts,
    ignore_index=True
)

# ------------------------------------------------------------
# 4. CROSS-CHECK AGAINST PREVIOUS ORDERING AUDIT
# ------------------------------------------------------------

reference = ordered_window_capacity[
    KEYS + [
        "windows_any_ordered_pair",
        "windows_any_three_step_change",
        "windows_three_distinct_order"
    ]
]

check = long_rotation_capacity.merge(
    reference,
    on=KEYS,
    how="left",
    validate="one_to_one"
)

assert check[
    "windows_any_ordered_pair"
].notna().all()

# Two alternating steps = ordered channel pair.
assert (
    check["windows_supporting_2steps"]
    == check["windows_any_ordered_pair"]
).all(), "Two-step validation failed"

# Three alternating steps = any valid 3-step motif.
assert (
    check["windows_supporting_3steps"]
    == check["windows_any_three_step_change"]
).all(), "Three-step validation failed"

# Three steps using E, U and H =
# a three-distinct-channel ordering.
assert (
    check["windows_3channel_3steps"]
    == check["windows_three_distinct_order"]
).all(), "Three-channel validation failed"

# ------------------------------------------------------------
# 5. ADDITIONAL VALIDATION
# ------------------------------------------------------------

assert long_rotation_capacity["user"].nunique() == 995

assert not long_rotation_capacity.duplicated(KEYS).any()

assert (
    long_rotation_capacity["max_feasible_rotation_steps"]
    <= long_rotation_capacity["window_workdays"]
).all()

assert (
    long_rotation_capacity["max_steps_using_3channels"]
    <= long_rotation_capacity["max_feasible_rotation_steps"]
).all()

assert (
    long_rotation_capacity["eligible_windows"]
    >= long_rotation_capacity["windows_supporting_2steps"]
).all()

for k in STEP_THRESHOLDS:

    assert (
        long_rotation_capacity[f"windows_supporting_{k}steps"]
        <= long_rotation_capacity["eligible_windows"]
    ).all()

    if k >= 3:
        assert (
            long_rotation_capacity[f"windows_3channel_{k}steps"]
            <= long_rotation_capacity[f"windows_supporting_{k}steps"]
        ).all()

# ------------------------------------------------------------
# 6. POPULATION COVERAGE SUMMARY
# ------------------------------------------------------------

def median_if_supported(series):
    selected = series[series >= 2]
    return (
        float(selected.median())
        if len(selected) > 0
        else np.nan
    )

long_rotation_summary = (
    long_rotation_capacity
    .groupby("window_workdays")
    .agg(
        users_with_windows=("user", "size"),
        users_with_rotation=(
            "max_feasible_rotation_steps",
            lambda x: int(x.ge(2).sum())
        ),
        users_with_5plus_steps=(
            "max_feasible_rotation_steps",
            lambda x: int(x.ge(5).sum())
        ),
        users_with_10plus_steps=(
            "max_feasible_rotation_steps",
            lambda x: int(x.ge(10).sum())
        ),
        users_with_15plus_steps=(
            "max_feasible_rotation_steps",
            lambda x: int(x.ge(15).sum())
        ),
        users_with_three_channels=(
            "max_steps_using_3channels",
            lambda x: int(x.ge(3).sum())
        ),
        users_with_3channel_5plus_steps=(
            "max_steps_using_3channels",
            lambda x: int(x.ge(5).sum())
        ),
        users_with_3channel_10plus_steps=(
            "max_steps_using_3channels",
            lambda x: int(x.ge(10).sum())
        ),
        median_user_best_steps=(
            "max_feasible_rotation_steps",
            median_if_supported
        ),
        maximum_supported_steps=(
            "max_feasible_rotation_steps", "max"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. COMPACT RESULTS
# ------------------------------------------------------------

print("=" * 80)
print("LONG-SEQUENCE CHANNEL ROTATION — FUTURE PERIOD")
print("=" * 80)

print(f"Multi-channel candidate users : {len(candidate_users):,}")
print(f"User-length records           : {len(long_rotation_capacity):,}")
print(
    "Maximum window length        :",
    int(long_rotation_summary["window_workdays"].max())
)

checkpoints = [
    3, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80, 100
]

selected = long_rotation_summary.loc[
    long_rotation_summary["window_workdays"].isin(
        checkpoints
    ),
    [
        "window_workdays",
        "users_with_windows",
        "users_with_rotation",
        "users_with_5plus_steps",
        "users_with_10plus_steps",
        "users_with_15plus_steps",
        "users_with_three_channels",
        "users_with_3channel_5plus_steps",
        "users_with_3channel_10plus_steps",
        "median_user_best_steps",
        "maximum_supported_steps"
    ]
]

print("\nLONG-ROTATION SUPPORT BY WINDOW LENGTH")
print("-" * 80)
print(selected.to_string(index=False))

# ------------------------------------------------------------
# 8. WINDOW COUNTS AT 30 WORKDAYS
# ------------------------------------------------------------

example_length = 30

example = long_rotation_capacity.loc[
    long_rotation_capacity["window_workdays"]
    == example_length
]

if not example.empty:

    print("\nWINDOW SUPPORT — 30 ELIGIBLE WORKDAYS")
    print("-" * 80)

    for k in STEP_THRESHOLDS:

        count = int(
            example[f"windows_supporting_{k}steps"].sum()
        )

        print(f"At least {k:>2} rotation steps: {count:,} windows")

    print("\nTHREE-CHANNEL SEQUENCES")

    for k in [3, 5, 8, 10, 15, 20, 30]:

        count = int(
            example[f"windows_3channel_{k}steps"].sum()
        )

        print(f"At least {k:>2} steps: {count:,} windows")

print("\nVALIDATION: PASS")
print("Only naturally active channel-days were used.")
print("No predefined rotation sequence imposed.")
print("No files saved.")


LONG-SEQUENCE CHANNEL ROTATION — FUTURE PERIOD
Multi-channel candidate users : 995
User-length records           : 103,395
Maximum window length        : 108

LONG-ROTATION SUPPORT BY WINDOW LENGTH
--------------------------------------------------------------------------------
 window_workdays  users_with_windows  users_with_rotation  users_with_5plus_steps  users_with_10plus_steps  users_with_15plus_steps  users_with_three_channels  users_with_3channel_5plus_steps  users_with_3channel_10plus_steps  median_user_best_steps  maximum_supported_steps
               3                 995                  994                       0                        0                        0                        273                                0                                 0                     3.0                        3
               5                 995                  995                     698                        0                        0                        288             


### 3.9 Long-Rotation Sequence Reconstruction and Validation

**Objective:** Independently reconstruct real channel-rotation sequences from observed user-day activity and validate the maximum lengths reported in Section 3.8.

**Analysis:**
- Select a user with median rotation support in a 30-workday window.
- Select a user with maximum rotation support.
- Select a user supporting a long sequence involving all three channels.
- Reconstruct chronological day-channel assignments.
- Verify that every selected channel is naturally active on its assigned day.
- Verify that selected days are distinct and consecutive steps use different channels.
- Count unused eligible days within each example window.

**Important:**
- The examples are selected from observed data, not predefined attack patterns.
- Unused days are potential no-injection days, not finalized Off-days.
- No injection, stealth validation, or scenario selection occurs.
- No files are saved.


In [13]:

# ============================================================
# 3.9 LONG-ROTATION SEQUENCE RECONSTRUCTION
# ============================================================

import numpy as np
import pandas as pd

FLAGS = ["email_active", "usb_active", "http_active"]
LABELS = "EUH"
WINDOW_LENGTH = 30

# ------------------------------------------------------------
# 1. INDEPENDENT SEQUENCE RECONSTRUCTION
# ------------------------------------------------------------

def reconstruct_sequence(flags, require_three=False):
    """
    Recover a longest alternating sequence.

    Each step selects one naturally active channel
    on a distinct eligible workday.

    Optionally require all three channels to appear.
    """

    states = {}

    for day_idx, daily_flags in enumerate(flags):

        previous = states.copy()
        updated = previous.copy()

        for channel in np.flatnonzero(daily_flags):

            channel = int(channel)
            bit = 1 << channel

            # Start a new sequence.
            key = (channel, bit)
            candidate = ((day_idx, channel),)

            if len(candidate) > len(updated.get(key, ())):
                updated[key] = candidate

            # Extend a sequence ending in a different channel.
            for (last, mask), path in previous.items():

                if last == channel:
                    continue

                new_key = (channel, mask | bit)
                candidate = path + ((day_idx, channel),)

                if len(candidate) > len(updated.get(new_key, ())):
                    updated[new_key] = candidate

        states = updated

    candidates = [
        path
        for (_, mask), path in states.items()
        if not require_three or mask == 7
    ]

    return max(candidates, key=len, default=())


def get_user_days(user):
    return (
        future_daily.loc[future_daily["user"] == user]
        .sort_values("day")
        .reset_index(drop=True)
    )


def validate_and_record(label, user, window, sequence, expected):

    flags = window[FLAGS].to_numpy()

    assert len(sequence) == expected
    assert len(sequence) <= len(window)

    for position, channel in sequence:
        assert flags[position, channel] == 1

    for previous, current in zip(sequence, sequence[1:]):
        assert current[0] > previous[0]
        assert current[1] != previous[1]

    pattern = " -> ".join(
        LABELS[channel] for _, channel in sequence
    )

    dated_steps = [
        f"{window.iloc[pos]['day']:%Y-%m-%d}:{LABELS[channel]}"
        for pos, channel in sequence
    ]

    print(f"\n{label}")
    print("-" * 72)
    print("User             :", user)
    print("Window start     :", window["day"].iloc[0].date())
    print("Window end       :", window["day"].iloc[-1].date())
    print("Eligible days    :", len(window))
    print("Rotation steps   :", len(sequence))
    print("Channel switches :", max(len(sequence) - 1, 0))
    print("Unselected days  :", len(window) - len(sequence))
    print("Pattern          :", pattern)
    print("Date/channel assignments:")
    print(" | ".join(dated_steps))

    return {
        "example": label,
        "user": user,
        "window_workdays": len(window),
        "window_start": window["day"].iloc[0],
        "window_end": window["day"].iloc[-1],
        "rotation_steps": len(sequence),
        "switches": max(len(sequence) - 1, 0),
        "unselected_days": len(window) - len(sequence),
        "pattern": pattern
    }


# ------------------------------------------------------------
# 2. SELECT DATA-DRIVEN EXAMPLES
# ------------------------------------------------------------

options = long_rotation_capacity.loc[
    long_rotation_capacity["window_workdays"] == WINDOW_LENGTH
].copy()

assert not options.empty

median_steps = options["max_feasible_rotation_steps"].median()

median_row = (
    options.assign(
        distance=(
            options["max_feasible_rotation_steps"] - median_steps
        ).abs()
    )
    .sort_values(["distance", "user"])
    .iloc[0]
)

maximum_row = (
    options.sort_values(
        ["max_feasible_rotation_steps", "user"],
        ascending=[False, True]
    )
    .iloc[0]
)

rotation_examples = []

# ------------------------------------------------------------
# 3. VALIDATE MAXIMUM ANY-CHANNEL SEQUENCES
# ------------------------------------------------------------

for label, row in [
    ("Median-support example", median_row),
    ("Maximum-support example", maximum_row)
]:

    user = row["user"]
    days = get_user_days(user)

    end = int(row["example_window_end_position"])
    start = end - WINDOW_LENGTH + 1

    assert start >= 0
    assert end < len(days)

    window = days.iloc[start:end + 1].reset_index(drop=True)

    sequence = reconstruct_sequence(
        window[FLAGS].to_numpy()
    )

    rotation_examples.append(
        validate_and_record(
            label,
            user,
            window,
            sequence,
            int(row["max_feasible_rotation_steps"])
        )
    )

# ------------------------------------------------------------
# 4. VALIDATE A THREE-CHANNEL LONG SEQUENCE
# ------------------------------------------------------------

three_options = options.loc[
    options["max_steps_using_3channels"] >= 3
].copy()

assert not three_options.empty

three_median = three_options[
    "max_steps_using_3channels"
].median()

three_row = (
    three_options.assign(
        distance=(
            three_options["max_steps_using_3channels"]
            - three_median
        ).abs()
    )
    .sort_values(["distance", "user"])
    .iloc[0]
)

three_user = three_row["user"]
three_days = get_user_days(three_user)

best_three_sequence = ()
best_three_window = None

# Search only this user's 30-workday windows.
for start in range(len(three_days) - WINDOW_LENGTH + 1):

    window = three_days.iloc[
        start:start + WINDOW_LENGTH
    ].reset_index(drop=True)

    sequence = reconstruct_sequence(
        window[FLAGS].to_numpy(),
        require_three=True
    )

    if len(sequence) > len(best_three_sequence):
        best_three_sequence = sequence
        best_three_window = window

assert len(best_three_sequence) == int(
    three_row["max_steps_using_3channels"]
)

assert len({
    channel for _, channel in best_three_sequence
}) == 3

rotation_examples.append(
    validate_and_record(
        "Three-channel example",
        three_user,
        best_three_window,
        best_three_sequence,
        int(three_row["max_steps_using_3channels"])
    )
)

# ------------------------------------------------------------
# 5. FINAL VALIDATION
# ------------------------------------------------------------

rotation_examples = pd.DataFrame(rotation_examples)

assert len(rotation_examples) == 3
assert rotation_examples["window_workdays"].eq(30).all()

print("\n" + "=" * 72)
print("LONG-SEQUENCE RECONSTRUCTION: VALIDATION PASS")
print("=" * 72)
print("All selected channel-days have natural activity.")
print("All steps use distinct days and alternate channels.")
print("No injection performed. No files saved.")



Median-support example
------------------------------------------------------------------------
User             : AAP1919
Window start     : 2011-01-04
Window end       : 2011-02-14
Eligible days    : 30
Rotation steps   : 17
Channel switches : 16
Unselected days  : 13
Pattern          : E -> U -> E -> U -> E -> U -> E -> U -> E -> U -> E -> U -> E -> U -> E -> U -> E
Date/channel assignments:
2011-01-04:E | 2011-01-05:U | 2011-01-06:E | 2011-01-07:U | 2011-01-10:E | 2011-01-20:U | 2011-01-21:E | 2011-01-31:U | 2011-02-01:E | 2011-02-02:U | 2011-02-03:E | 2011-02-04:U | 2011-02-07:E | 2011-02-09:U | 2011-02-10:E | 2011-02-11:U | 2011-02-14:E

Maximum-support example
------------------------------------------------------------------------
User             : ABD3426
Window start     : 2011-01-03
Window end       : 2011-02-11
Eligible days    : 30
Rotation steps   : 30
Channel switches : 29
Unselected days  : 0
Pattern          : E -> U -> E -> U -> E -> U -> E -> U -> E -> U -> E -> U 


### 3.10 Rotation Cadence and Off-Day Feasibility

**Objective:** Evaluate how naturally supported channel-rotation sequences change when non-injection days are allowed or required between selected attack steps.

**Exploratory Window Durations:**
5, 10, 20, 30, 40, and 50 eligible workdays.

These durations are candidates for analysis, not approved attack durations.

**Cadence Comparisons:**
- **Gap 0:** No minimum separation between selected steps.
- **Gap 1:** At least one eligible workday without injection between consecutive selected steps.
- **Gap 2:** At least two eligible workdays without injection between consecutive selected steps.

**Strict Channel-Day Rules:**
- Only naturally multi-channel users are considered.
- Each selected step uses a channel naturally active on that exact user-day.
- Each selected step occupies a different eligible workday.
- Consecutive selected steps must use different channels.
- Each analytical sequence selects at most one injection channel per day.
- Unselected days may still contain natural benign activity.

**Analysis:**
- Determine the maximum supported rotation steps for every window under each cadence.
- Calculate maximum supported switches.
- Compare windows supporting at least 2, 5, and 10 steps.
- Separately assess sequences using all three channels.
- Measure unused eligible days when using the maximum supported steps.
- Validate Gap 0 results against the previous long-rotation audit.

**Interpretation:**
- Gap 0, Gap 1, and Gap 2 are exploratory comparisons, not mandatory constraints.
- Unselected days are potential Off-days, not finalized attack schedules.
- A longer supported sequence is not necessarily stealthier.
- No attack volume, payload, or injection is generated.

**Expected In-Memory Outputs:**
- `rotation_cadence_capacity`
- `rotation_cadence_summary`

No files are saved at this stage.


In [14]:

# ============================================================
# 3.10 ROTATION CADENCE & OFF-DAY FEASIBILITY
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

CANDIDATE_DURATIONS = [5, 10, 20, 30, 40, 50]

# Minimum number of unselected eligible workdays
# BETWEEN consecutive selected rotation steps.
CADENCE_GAPS = [0, 1, 2]

FLAGS = ["email_active", "usb_active", "http_active"]
KEYS = ["user", "window_workdays"]

candidate_users = set(
    user_profile.loc[
        user_profile["active_channel_count"] >= 2,
        "user"
    ]
)

assert len(candidate_users) == 995

candidate_daily = (
    future_daily.loc[
        future_daily["user"].isin(candidate_users)
    ]
    .sort_values(["user", "day"])
)

# ------------------------------------------------------------
# 2. DYNAMIC PROGRAMMING FOR ALL WINDOW POSITIONS
# ------------------------------------------------------------

def calculate_cadence_windows(flags, durations, gaps):
    """
    For every supported window length and start position,
    find the maximum alternating sequence under each gap.

    DP state:
      [window_start, last_channel, used_channel_mask]

    Channel masks:
      Email=1, USB=2, HTTP=4, all three=7.

    Each selected day must have natural activity in
    its selected channel.

    A gap of g requires at least g unselected eligible
    days BETWEEN consecutive selected steps.
    """

    n = len(flags)

    valid_lengths = [
        L for L in durations if L <= n
    ]

    output = {}

    for gap in gaps:

        results = {
            L: {
                "any": np.zeros(n - L + 1, dtype=np.int16),
                "three": np.zeros(n - L + 1, dtype=np.int16)
            }
            for L in valid_lengths
        }

        # -1 means an unreachable DP state.
        cumulative = np.full(
            (n, 3, 8), -1, dtype=np.int16
        )

        history = []

        for end in range(n):

            current = np.full_like(cumulative, -1)

            # A preceding selected day must be at or
            # before end-gap-1.
            previous_end = end - gap - 1

            previous = (
                history[previous_end]
                if previous_end >= 0
                else None
            )

            active_channels = np.flatnonzero(
                flags[end]
            )

            for channel in active_channels:

                channel = int(channel)
                bit = 1 << channel

                # A new sequence may start on this day.
                # Only windows starting at/before 'end'
                # are valid.
                current[
                    :end + 1, channel, bit
                ] = 1

                if previous is None:
                    continue

                other_a = (channel + 1) % 3
                other_b = (channel + 2) % 3

                # A new step must differ from the
                # previous selected channel.
                previous_other = np.maximum(
                    previous[:, other_a, :],
                    previous[:, other_b, :]
                )

                for old_mask in range(1, 8):

                    new_mask = old_mask | bit

                    old_lengths = previous_other[
                        :end + 1, old_mask
                    ]

                    candidate_lengths = np.where(
                        old_lengths >= 1,
                        old_lengths + 1,
                        -1
                    )

                    destination = current[
                        :end + 1,
                        channel,
                        new_mask
                    ]

                    np.maximum(
                        destination,
                        candidate_lengths,
                        out=destination
                    )

            # Keep either an earlier completed sequence
            # or one ending on this day.
            cumulative = np.maximum(
                cumulative,
                current
            )

            history.append(cumulative)

            # Record the exact result of each window
            # ending today, for each selected duration.
            for L in valid_lengths:

                if end + 1 < L:
                    continue

                start = end - L + 1

                best_any = int(
                    cumulative[start].max()
                )

                best_three = int(
                    cumulative[start, :, 7].max()
                )

                results[L]["any"][start] = max(
                    best_any, 0
                )

                results[L]["three"][start] = max(
                    best_three, 0
                )

        output[gap] = results

    return output


# ------------------------------------------------------------
# 3. EVALUATE ALL MULTI-CHANNEL USERS
# ------------------------------------------------------------

cadence_records = []

for user, group in candidate_daily.groupby(
    "user", sort=False
):

    flags = group[FLAGS].to_numpy(
        dtype=np.int8
    )

    n = len(flags)

    user_results = calculate_cadence_windows(
        flags,
        CANDIDATE_DURATIONS,
        CADENCE_GAPS
    )

    for gap, length_results in user_results.items():

        for L, values in length_results.items():

            best_any = values["any"]
            best_three = values["three"]

            n_windows = len(best_any)

            # Days not selected when the sequence
            # uses the maximum supported steps.
            unselected_days = L - best_any

            record = {
                "user": user,
                "window_workdays": L,
                "minimum_interstep_offdays": gap,
                "eligible_windows": n_windows,

                "maximum_rotation_steps": int(
                    best_any.max()
                ),

                "maximum_switches": int(
                    max(best_any.max() - 1, 0)
                ),

                "maximum_3channel_steps": int(
                    best_three.max()
                ),

                "median_window_max_steps": float(
                    np.median(best_any)
                ),

                "median_unselected_days_at_window_max":
                    float(np.median(unselected_days)),

                "unselected_days_at_user_best": int(
                    L - best_any.max()
                ),

                "windows_with_2plus_steps": int(
                    (best_any >= 2).sum()
                ),

                "windows_with_5plus_steps": int(
                    (best_any >= 5).sum()
                ),

                "windows_with_10plus_steps": int(
                    (best_any >= 10).sum()
                ),

                "windows_with_15plus_steps": int(
                    (best_any >= 15).sum()
                ),

                "windows_with_3channel_rotation": int(
                    (best_three >= 3).sum()
                ),

                "windows_with_3channel_5plus_steps": int(
                    (best_three >= 5).sum()
                ),

                "windows_with_3channel_10plus_steps": int(
                    (best_three >= 10).sum()
                )
            }

            cadence_records.append(record)

rotation_cadence_capacity = pd.DataFrame(
    cadence_records
).sort_values(
    ["window_workdays", "minimum_interstep_offdays", "user"]
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. CROSS-CHECK GAP 0 AGAINST SECTION 3.8
# ------------------------------------------------------------

gap_zero = rotation_cadence_capacity.loc[
    rotation_cadence_capacity[
        "minimum_interstep_offdays"
    ] == 0
]

reference = long_rotation_capacity.loc[
    long_rotation_capacity["window_workdays"].isin(
        CANDIDATE_DURATIONS
    ),
    KEYS + [
        "max_feasible_rotation_steps",
        "max_steps_using_3channels",
        "windows_supporting_2steps",
        "windows_supporting_5steps",
        "windows_supporting_10steps",
        "windows_3channel_3steps"
    ]
]

check = gap_zero.merge(
    reference,
    on=KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True
)

assert check["_merge"].eq("both").all()

comparisons = [
    ("maximum_rotation_steps",
     "max_feasible_rotation_steps"),

    ("maximum_3channel_steps",
     "max_steps_using_3channels"),

    ("windows_with_2plus_steps",
     "windows_supporting_2steps"),

    ("windows_with_5plus_steps",
     "windows_supporting_5steps"),

    ("windows_with_10plus_steps",
     "windows_supporting_10steps"),

    ("windows_with_3channel_rotation",
     "windows_3channel_3steps")
]

for actual, expected in comparisons:

    assert (
        check[actual] == check[expected]
    ).all(), f"Gap 0 mismatch: {actual}"

# ------------------------------------------------------------
# 5. CADENCE MONOTONICITY VALIDATION
# ------------------------------------------------------------

assert not rotation_cadence_capacity.duplicated(
    KEYS + ["minimum_interstep_offdays"]
).any()

assert rotation_cadence_capacity[
    "maximum_rotation_steps"
].le(
    rotation_cadence_capacity["window_workdays"]
).all()

assert (
    rotation_cadence_capacity[
        "maximum_3channel_steps"
    ]
    <= rotation_cadence_capacity[
        "maximum_rotation_steps"
    ]
).all()

assert (
    rotation_cadence_capacity["windows_with_3channel_rotation"]
    <= rotation_cadence_capacity["windows_with_2plus_steps"]
).all()

# With g unselected days required between steps,
# at most floor((L+g)/(g+1)) steps can be selected.
maximum_possible = (
    (
        rotation_cadence_capacity["window_workdays"]
        + rotation_cadence_capacity[
            "minimum_interstep_offdays"
        ]
    )
    // (
        rotation_cadence_capacity[
            "minimum_interstep_offdays"
        ] + 1
    )
)

assert (
    rotation_cadence_capacity["maximum_rotation_steps"]
    <= maximum_possible
).all()

metrics_to_compare = [
    "maximum_rotation_steps",
    "maximum_3channel_steps",
    "windows_with_2plus_steps",
    "windows_with_5plus_steps",
    "windows_with_10plus_steps",
    "windows_with_3channel_rotation"
]

for metric in metrics_to_compare:

    wide = rotation_cadence_capacity.pivot(
        index=KEYS,
        columns="minimum_interstep_offdays",
        values=metric
    )

    assert (
        (wide[0] >= wide[1])
        & (wide[1] >= wide[2])
    ).all(), f"Non-monotonic cadence results: {metric}"

# ------------------------------------------------------------
# 6. POPULATION SUMMARY
# ------------------------------------------------------------

rotation_cadence_summary = (
    rotation_cadence_capacity
    .groupby(
        ["window_workdays", "minimum_interstep_offdays"]
    )
    .agg(
        users_with_windows=("user", "size"),

        users_with_rotation=(
            "maximum_rotation_steps",
            lambda x: int(x.ge(2).sum())
        ),

        users_with_5plus_steps=(
            "maximum_rotation_steps",
            lambda x: int(x.ge(5).sum())
        ),

        users_with_10plus_steps=(
            "maximum_rotation_steps",
            lambda x: int(x.ge(10).sum())
        ),

        users_with_3channel_rotation=(
            "maximum_3channel_steps",
            lambda x: int(x.ge(3).sum())
        ),

        median_user_best_steps=(
            "maximum_rotation_steps",
            "median"
        ),

        median_unselected_days_at_user_best=(
            "unselected_days_at_user_best",
            "median"
        ),

        total_windows_with_rotation=(
            "windows_with_2plus_steps",
            "sum"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 7. COMPACT RESULTS
# ------------------------------------------------------------

print("=" * 82)
print("ROTATION CADENCE & OFF-DAY FEASIBILITY")
print("=" * 82)

print(f"Multi-channel users analyzed : {len(candidate_users):,}")
print(f"Candidate durations          : {CANDIDATE_DURATIONS}")
print(f"Minimum inter-step gaps      : {CADENCE_GAPS}")
print(f"User-duration-gap records    : {len(rotation_cadence_capacity):,}")

print("\nROTATION SUPPORT BY DURATION AND CADENCE")
print("-" * 82)

display_columns = [
    "window_workdays",
    "minimum_interstep_offdays",
    "users_with_windows",
    "users_with_rotation",
    "users_with_5plus_steps",
    "users_with_10plus_steps",
    "users_with_3channel_rotation",
    "median_user_best_steps",
    "median_unselected_days_at_user_best"
]

print(
    rotation_cadence_summary[
        display_columns
    ].to_string(index=False)
)

print("\nCROSS-CHECK WITH LONG ROTATION: PASS")
print("CADENCE MONOTONICITY: PASS")
print("VALIDATION: PASS")
print("No cadence or Off-day rule has been approved.")
print("No files saved.")


ROTATION CADENCE & OFF-DAY FEASIBILITY
Multi-channel users analyzed : 995
Candidate durations          : [5, 10, 20, 30, 40, 50]
Minimum inter-step gaps      : [0, 1, 2]
User-duration-gap records    : 17,799

ROTATION SUPPORT BY DURATION AND CADENCE
----------------------------------------------------------------------------------
 window_workdays  minimum_interstep_offdays  users_with_windows  users_with_rotation  users_with_5plus_steps  users_with_10plus_steps  users_with_3channel_rotation  median_user_best_steps  median_unselected_days_at_user_best
               5                          0                 995                  995                     698                        0                           288                     5.0                                  0.0
               5                          1                 995                  995                       0                        0                           281                     3.0                              


### 3.11 Channel Contribution and Rotation Spacing Audit

**Objective:** Examine channel participation and temporal spacing in naturally supported long-rotation sequences.

**Exploratory Window Durations:**
5, 10, 20, 30, 40, and 50 eligible workdays.

**Part A — Channel Contribution**
- Reconstruct a maximum-length rotation sequence for each user and candidate duration.
- Count the selected Email, USB, and HTTP steps.
- Measure dominant and least-used channel contributions.
- Count distinct directed channel transitions.
- Examine sequences using all three channels.
- Identify cases where one channel appears only once.

**Part B — Rotation Spacing**
- Examine minimum inter-step gaps of 0, 1, and 2 eligible workdays.
- Count unselected eligible workdays.
- Separate unselected days before, between, and after the selected steps.
- Measure median and longest internal gaps.
- Compare reconstructed results with the previously calculated global cadence maxima.

**Sampling Scope:**
One data-driven representative window per user and duration, selected from the maximum Gap-0 support. Alternative cadence schedules are reconstructed inside that same window.

**Strict Rules:**
- Only naturally multi-channel users are considered.
- Every selected channel must be active naturally on its exact assigned day.
- Each step occupies a distinct eligible workday.
- Consecutive steps must use different channels.
- Ineligible days are excluded from window lengths.
- Unselected eligible days have no attack injection in the reconstructed sequence.

**Interpretation:**
Results describe valid analytical sequences, not final attack schedules. When several optimal sequences exist, a reconstructed example does not uniquely represent all possible channel distributions.

No scenario durations, cadence thresholds, or channel-balance constraints are approved by this audit.

**Outputs in Memory:**
- `rotation_structure_detail`
- `rotation_structure_summary`

No files are saved.


In [15]:

# ============================================================
# 3.11 CHANNEL CONTRIBUTION & ROTATION SPACING AUDIT
# ============================================================

import numpy as np
import pandas as pd

DURATIONS = [5, 10, 20, 30, 40, 50]
GAPS = [0, 1, 2]

FLAGS = ["email_active", "usb_active", "http_active"]
CHANNELS = ["E", "U", "H"]
KEYS = ["user", "window_workdays"]

# ------------------------------------------------------------
# 1. PREPARE VALIDATED REFERENCES
# ------------------------------------------------------------

reference = long_rotation_capacity.loc[
    long_rotation_capacity["window_workdays"].isin(DURATIONS)
].copy()

cadence_reference = rotation_cadence_capacity.set_index(
    KEYS + ["minimum_interstep_offdays"],
    verify_integrity=True
)

user_days = {
    user: group.sort_values("day").reset_index(drop=True)
    for user, group in future_daily.loc[
        future_daily["user"].isin(reference["user"].unique())
    ].groupby("user")
}

assert reference["user"].nunique() == 995
assert not reference.duplicated(KEYS).any()

# ------------------------------------------------------------
# 2. RECONSTRUCT VALID ROTATION SEQUENCES
# ------------------------------------------------------------

def reconstruct_with_gap(flags, min_gap=0):
    """
    Return:
      - longest naturally supported alternating sequence
      - longest sequence involving all three channels

    Sequence entries are (window_day_position, channel_index).

    DP states are (last_channel, used_channel_mask).

    The required gap is measured in eligible workdays
    without injection between selected steps.
    """

    history = []

    def keep_longer(states, key, path):
        if len(path) > len(states.get(key, ())):
            states[key] = path

    for day_index, daily_flags in enumerate(flags):

        current = (
            history[-1].copy()
            if history else {}
        )

        previous_index = day_index - min_gap - 1

        previous = (
            history[previous_index]
            if previous_index >= 0
            else {}
        )

        for channel in np.flatnonzero(daily_flags):

            channel = int(channel)
            bit = 1 << channel

            # Start a sequence at this naturally active day.
            keep_longer(
                current,
                (channel, bit),
                ((day_index, channel),)
            )

            # Extend a sequence from a sufficiently
            # earlier day, switching channels.
            for (last_channel, old_mask), path in previous.items():

                if last_channel == channel:
                    continue

                new_key = (
                    channel,
                    old_mask | bit
                )

                new_path = path + (
                    (day_index, channel),
                )

                keep_longer(
                    current,
                    new_key,
                    new_path
                )

        history.append(current)

    if not history:
        return (), ()

    states = history[-1]

    # Prefer more distinct channels only when the
    # maximum sequence length is tied.
    # This is a reconstruction tie-break, not an
    # attack constraint.
    best_any = max(
        states.items(),
        key=lambda item: (
            len(item[1]),
            item[0][1].bit_count()
        ),
        default=(None, ())
    )[1]

    best_three = max(
        (
            path
            for (last_channel, mask), path
            in states.items()
            if mask == 7
        ),
        key=len,
        default=()
    )

    return best_any, best_three


# ------------------------------------------------------------
# 3. DESCRIBE CHANNEL CONTRIBUTION AND SPACING
# ------------------------------------------------------------

def describe_sequence(
    user,
    window,
    sequence,
    gap,
    variant,
    global_best
):

    if not sequence:
        return None

    length = len(window)
    flags = window[FLAGS].to_numpy(dtype=np.int8)

    positions = np.array(
        [position for position, _ in sequence],
        dtype=np.int32
    )

    selected_channels = np.array(
        [channel for _, channel in sequence],
        dtype=np.int8
    )

    assert np.all(
        flags[positions, selected_channels] == 1
    )

    assert np.all(np.diff(positions) > 0)

    assert np.all(
        np.diff(positions) >= gap + 1
    )

    assert np.all(
        selected_channels[1:] != selected_channels[:-1]
    )

    counts = np.bincount(
        selected_channels,
        minlength=3
    )

    used_counts = counts[counts > 0]

    internal_gaps = np.diff(positions) - 1

    leading_unselected = int(positions[0])

    trailing_unselected = int(
        length - 1 - positions[-1]
    )

    internal_unselected = int(
        internal_gaps.sum()
    )

    total_unselected = length - len(sequence)

    assert (
        leading_unselected
        + internal_unselected
        + trailing_unselected
        == total_unselected
    )

    transition_counts = {
        f"transitions_{a}_{b}": 0
        for a in CHANNELS
        for b in CHANNELS
        if a != b
    }

    for first, second in zip(
        selected_channels[:-1],
        selected_channels[1:]
    ):
        transition_counts[
            f"transitions_{CHANNELS[first]}_{CHANNELS[second]}"
        ] += 1

    observed_active_days = flags.sum(axis=0)

    record = {
        "user": user,
        "window_workdays": length,
        "minimum_interstep_offdays": gap,
        "sequence_variant": variant,

        "window_start": pd.Timestamp(
            window["day"].iloc[0]
        ),
        "window_end": pd.Timestamp(
            window["day"].iloc[-1]
        ),

        "rotation_steps": len(sequence),
        "channel_switches": len(sequence) - 1,
        "used_channel_count": len(used_counts),

        "email_steps": int(counts[0]),
        "usb_steps": int(counts[1]),
        "http_steps": int(counts[2]),

        "observed_email_active_days": int(
            observed_active_days[0]
        ),
        "observed_usb_active_days": int(
            observed_active_days[1]
        ),
        "observed_http_active_days": int(
            observed_active_days[2]
        ),

        "dominant_channel_share": float(
            counts.max() / len(sequence)
        ),

        "least_used_channel_steps": int(
            used_counts.min()
        ),

        "one_step_channel_present": bool(
            (used_counts == 1).any()
        ),

        "distinct_transition_types": int(
            sum(v > 0 for v in transition_counts.values())
        ),

        "total_unselected_days": total_unselected,
        "leading_unselected_days": leading_unselected,
        "internal_unselected_days": internal_unselected,
        "trailing_unselected_days": trailing_unselected,

        "median_internal_gap": float(
            np.median(internal_gaps)
            if len(internal_gaps) else 0
        ),

        "longest_internal_gap": int(
            internal_gaps.max()
            if len(internal_gaps) else 0
        ),

        "zero_gap_transitions": int(
            (internal_gaps == 0).sum()
        ),

        "one_day_gap_transitions": int(
            (internal_gaps == 1).sum()
        ),

        "two_plus_day_gap_transitions": int(
            (internal_gaps >= 2).sum()
        ),

        "global_best_steps": int(global_best),

        "matches_global_best": bool(
            len(sequence) == global_best
        ),

        # Compact reconstruction retained in memory.
        "sequence_pattern": "".join(
            CHANNELS[channel]
            for channel in selected_channels
        ),

        "selected_day_positions": ",".join(
            map(str, positions)
        )
    }

    record.update(transition_counts)

    assert (
        record["rotation_steps"]
        <= global_best
    )

    assert (
        sum(transition_counts.values())
        == record["channel_switches"]
    )

    return record


# ------------------------------------------------------------
# 4. ANALYZE REPRESENTATIVE WINDOWS
# ------------------------------------------------------------

records = []

for row in reference.itertuples(index=False):

    user = row.user
    length = int(row.window_workdays)

    days = user_days[user]

    end = int(row.example_window_end_position)
    start = end - length + 1

    assert 0 <= start <= end < len(days)

    window = days.iloc[
        start:end + 1
    ].reset_index(drop=True)

    flags = window[FLAGS].to_numpy(
        dtype=np.int8
    )

    for gap in GAPS:

        key = (user, length, gap)

        global_record = cadence_reference.loc[key]

        global_any = int(
            global_record["maximum_rotation_steps"]
        )

        global_three = int(
            global_record["maximum_3channel_steps"]
        )

        best_any, best_three = reconstruct_with_gap(
            flags,
            min_gap=gap
        )

        # Gap 0 must reproduce the maximum already
        # established in the full-window audit.
        if gap == 0:

            assert len(best_any) == int(
                row.max_feasible_rotation_steps
            )

            assert len(best_any) == global_any

        assert len(best_any) <= global_any
        assert len(best_three) <= global_three

        if best_any:

            records.append(
                describe_sequence(
                    user,
                    window,
                    best_any,
                    gap,
                    "maximum_rotation",
                    global_any
                )
            )

        # Optional alternative: the longest sequence
        # using all three channels in this same window.
        # It may be shorter than the maximum sequence.
        if len(best_three) >= 3:

            records.append(
                describe_sequence(
                    user,
                    window,
                    best_three,
                    gap,
                    "three_channel_alternative",
                    global_three
                )
            )

rotation_structure_detail = pd.DataFrame(
    records
).sort_values(
    [
        "window_workdays",
        "minimum_interstep_offdays",
        "sequence_variant",
        "user"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 5. VALIDATE RESULTS
# ------------------------------------------------------------

DETAIL_KEYS = KEYS + [
    "minimum_interstep_offdays",
    "sequence_variant"
]

assert not rotation_structure_detail.duplicated(
    DETAIL_KEYS
).any()

main_sequences = rotation_structure_detail.loc[
    rotation_structure_detail["sequence_variant"]
    == "maximum_rotation"
].copy()

assert len(main_sequences) == len(reference) * 3

assert (
    main_sequences["rotation_steps"]
    <= main_sequences["window_workdays"]
).all()

assert (
    main_sequences["rotation_steps"]
    == main_sequences["channel_switches"] + 1
).all()

assert (
    main_sequences["matches_global_best"]
    [main_sequences["minimum_interstep_offdays"] == 0]
).all()

assert (
    rotation_structure_detail[
        "total_unselected_days"
    ]
    == rotation_structure_detail[
        "leading_unselected_days"
    ]
    + rotation_structure_detail[
        "internal_unselected_days"
    ]
    + rotation_structure_detail[
        "trailing_unselected_days"
    ]
).all()

assert (
    rotation_structure_detail[
        "email_steps"
    ]
    + rotation_structure_detail[
        "usb_steps"
    ]
    + rotation_structure_detail[
        "http_steps"
    ]
    == rotation_structure_detail["rotation_steps"]
).all()

# Enforcing a larger gap cannot increase the
# maximum number of steps within the same window.
step_wide = main_sequences.pivot(
    index=KEYS,
    columns="minimum_interstep_offdays",
    values="rotation_steps"
)

assert (
    (step_wide[0] >= step_wide[1])
    & (step_wide[1] >= step_wide[2])
).all()

three_sequences = rotation_structure_detail.loc[
    rotation_structure_detail["sequence_variant"]
    == "three_channel_alternative"
]

assert three_sequences[
    "used_channel_count"
].eq(3).all()

# ------------------------------------------------------------
# 6. POPULATION SUMMARY
# ------------------------------------------------------------

rotation_structure_summary = (
    rotation_structure_detail
    .groupby(
        [
            "window_workdays",
            "minimum_interstep_offdays",
            "sequence_variant"
        ]
    )
    .agg(
        users=("user", "nunique"),

        median_steps=(
            "rotation_steps", "median"
        ),

        median_used_channels=(
            "used_channel_count", "median"
        ),

        users_using_three_channels=(
            "used_channel_count",
            lambda x: int(x.eq(3).sum())
        ),

        median_dominant_share=(
            "dominant_channel_share", "median"
        ),

        median_least_used_channel_steps=(
            "least_used_channel_steps", "median"
        ),

        users_with_one_step_channel=(
            "one_step_channel_present",
            "sum"
        ),

        median_distinct_transitions=(
            "distinct_transition_types", "median"
        ),

        median_unselected_days=(
            "total_unselected_days", "median"
        ),

        median_internal_unselected=(
            "internal_unselected_days", "median"
        ),

        median_longest_internal_gap=(
            "longest_internal_gap", "median"
        ),

        users_matching_global_best=(
            "matches_global_best", "sum"
        )
    )
    .reset_index()
)

rotation_structure_summary[
    "global_best_match_pct"
] = (
    100
    * rotation_structure_summary["users_matching_global_best"]
    / rotation_structure_summary["users"]
).round(1)

rotation_structure_summary[
    "median_dominant_share_pct"
] = (
    100
    * rotation_structure_summary["median_dominant_share"]
).round(1)

# ------------------------------------------------------------
# 7. COMPACT RESULTS
# ------------------------------------------------------------

print("=" * 88)
print("CHANNEL CONTRIBUTION & ROTATION SPACING AUDIT")
print("=" * 88)

print(f"Candidate durations         : {DURATIONS}")
print(f"Minimum inter-step gaps     : {GAPS}")
print(f"Users analyzed              : {reference['user'].nunique():,}")
print(f"User-duration pairs         : {len(reference):,}")
print(f"Reconstructed sequence rows : {len(rotation_structure_detail):,}")

main_summary = rotation_structure_summary.loc[
    rotation_structure_summary["sequence_variant"]
    == "maximum_rotation"
]

print("\nMAXIMUM ROTATION — CHANNEL CONTRIBUTION & SPACING")
print("-" * 88)

print(
    main_summary[
        [
            "window_workdays",
            "minimum_interstep_offdays",
            "users",
            "median_steps",
            "users_using_three_channels",
            "median_dominant_share_pct",
            "median_internal_unselected",
            "median_longest_internal_gap",
            "global_best_match_pct"
        ]
    ].to_string(index=False)
)

print("\nTHREE-CHANNEL ALTERNATIVES — GAP 0")
print("-" * 88)

three_summary = rotation_structure_summary.loc[
    (
        rotation_structure_summary["sequence_variant"]
        == "three_channel_alternative"
    )
    & (
        rotation_structure_summary[
            "minimum_interstep_offdays"
        ] == 0
    )
]

print(
    three_summary[
        [
            "window_workdays",
            "users",
            "median_steps",
            "median_dominant_share_pct",
            "median_least_used_channel_steps",
            "users_with_one_step_channel",
            "median_internal_unselected"
        ]
    ].to_string(index=False)
)

print("\nVALIDATION: PASS")
print("Every selected channel was naturally active on its exact day.")
print("All rotations and inter-step gaps were validated.")
print("No scenario rules approved. No injections or files saved.")


C:\Users\hp\AppData\Local\Temp\ipykernel_14080\2199763936.py:23: Pandas4Warning: The 'verify_integrity' keyword in DataFrame.set_index is deprecated and will be removed in a future version. Directly check the result.index.is_unique instead.
  cadence_reference = rotation_cadence_capacity.set_index(


CHANNEL CONTRIBUTION & ROTATION SPACING AUDIT
Candidate durations         : [5, 10, 20, 30, 40, 50]
Minimum inter-step gaps     : [0, 1, 2]
Users analyzed              : 995
User-duration pairs         : 5,933
Reconstructed sequence rows : 22,296

MAXIMUM ROTATION — CHANNEL CONTRIBUTION & SPACING
----------------------------------------------------------------------------------------
 window_workdays  minimum_interstep_offdays  users  median_steps  users_using_three_channels  median_dominant_share_pct  median_internal_unselected  median_longest_internal_gap  global_best_match_pct
               5                          0    995           5.0                         219                       60.0                         0.0                          0.0                  100.0
               5                          1    995           2.0                          91                       50.0                         2.0                          1.0                   47.3
             


## 5. Pre-Export Numerical and Logical Consistency Audit

**Objective:** Independently verify the consistency of all in-memory Notebook 02 outputs before export.

**Validation Scope:**
- Source and population integrity
- Temporal window arithmetic
- Channel-state partitions
- Distinct-day and ordered-channel support
- Exclusive and flexible rotation relationships
- Channel recurrence and support percentages
- Long-sequence rotation calculations
- Cadence constraints and monotonicity
- Exact reconstructed day-channel assignments
- Population and structural summary reconciliation
- Independent exhaustive verification of sampled windows

**Acceptance Criteria:**
- Every required check must pass.
- Any failure blocks final export.
- All discrepancies must be investigated before generating reports.

**Restrictions:**
- No baseline modification.
- No original Future modification.
- No attack injection.
- No output files saved.

**Expected Result:** A consolidated PASS/FAIL report with explicit failure reasons.


In [17]:

# ============================================================
# 5.1 PRE-EXPORT CONSISTENCY AUDIT — V2
# ============================================================

from pathlib import Path

AUDIT_SCRIPT = (
    PROJECT_ROOT
    / "MUBSIR_NB02_PreExport_Consistency_Audit_v2.py"
)

assert AUDIT_SCRIPT.is_file(), (
    f"Audit script not found: {AUDIT_SCRIPT}"
)

exec(
    compile(
        AUDIT_SCRIPT.read_text(encoding="utf-8"),
        str(AUDIT_SCRIPT),
        "exec"
    ),
    globals()
)


NOTEBOOK 02 | PRE-EXPORT NUMERICAL + LOGICAL CONSISTENCY AUDIT
PASS | 01 Scope / daily channel flags | 3995 baseline users; 3737 active Future; 995 multi-channel
PASS | 02 Temporal runs and calendar segments | Temporal user counts and each calendar segment reconcile with source days
PASS | 03 User-length windows / calendar partition | 387,162 user-length pairs; 20,436,421 candidate windows
PASS | 04 Exact channel-state partition | 8 channel states and all derived unions reconcile
PASS | 05 Distinct-day / ordered / exclusive support | Distinct-day, ordering, Hall conditions and exclusive subsets reconcile
PASS | 06 Recurrence / support percentages | 8 percentage columns within 0–100%
PASS | 07 Long-rotation support arithmetic | 103,395 multi-channel user-length rows valid
PASS | 08 Gap 0 parity / cadence bounds and monotonicity | 17,799 duration/gap records and Gap-0 parity verified
PASS | 09 Every reconstructed path vs actual user-day flags | 22,296 paths checked against exact day/chan


## 6. Validated Results Export

**Objective:** Export the audited Notebook 02 results without changing source datasets.

**Output 1 — `user_window_length_audit.parquet`**

One row per user and supported eligible-workday window length.

Includes temporal availability, observed channel states, distinct-day support, channel ordering, recurrence, and long-rotation results where applicable.

Long-rotation fields are null for users outside the multi-channel candidate population. Null means **not applicable**, not zero feasibility.

**Output 2 — `rotation_feasibility_audit.parquet`**

One row per reconstructed user-duration-cadence-sequence variant.

Includes representative window dates, selected sequence positions and channel order, channel contributions, internal gaps, unused days, and the corresponding global cadence capacity.

**Scope:**
- Window lengths are measured in eligible workdays.
- Candidate durations and cadence gaps remain exploratory.
- Reconstructed sequences are analytical examples, not approved injection schedules.
- Original Baseline and Future datasets remain read-only.

**Acceptance:**
Both outputs must pass key, row-count, relationship, and Parquet verification checks before finalizing.


In [19]:

# ============================================================
# 6.1 VALIDATED CONSOLIDATION AND EXPORT — CORRECTED
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

KEYS = ["user", "window_workdays"]
CADENCE_KEYS = KEYS + ["minimum_interstep_offdays"]
STRUCTURE_KEYS = CADENCE_KEYS + ["sequence_variant"]

REPORT_DIR_02 = (
    PROJECT_ROOT / "reports" / "02_Temporal_Window_Audit"
)

# ------------------------------------------------------------
# 1. STANDARDIZE KEY DTYPES WITHOUT CHANGING SOURCE TABLES
# ------------------------------------------------------------

def standardize_keys(frame, include_gap=False):
    """
    Return an independent copy with consistent key types.

    This avoids false index mismatches caused by
    int32 versus int64 window lengths.
    """

    result = frame.copy()

    assert "user" in result.columns
    assert "window_workdays" in result.columns

    result["window_workdays"] = pd.to_numeric(
        result["window_workdays"],
        errors="raise"
    ).astype("int64")

    assert result["window_workdays"].gt(0).all()

    if include_gap:

        assert "minimum_interstep_offdays" in result.columns

        result["minimum_interstep_offdays"] = (
            pd.to_numeric(
                result["minimum_interstep_offdays"],
                errors="raise"
            ).astype("int64")
        )

    return result


# ------------------------------------------------------------
# 2. CONSOLIDATE ALL USER-WINDOW-LENGTH TABLES
# ------------------------------------------------------------

source_tables = [
    ("window_capacity", window_length_capacity),
    ("channel_availability", channel_window_capacity),
    ("distinct_day", distinct_day_window_capacity),
    ("ordered_rotation", ordered_window_capacity),
    ("exclusive_rotation", exclusive_order_capacity),
    ("support_strength", window_support_strength),
    ("channel_recurrence", channel_recurrence_capacity),
]

indexed_tables = []

for source_name, original in source_tables:

    frame = standardize_keys(original)

    assert not frame.columns.duplicated().any(), source_name
    assert not frame.duplicated(KEYS).any(), source_name

    indexed = (
        frame.set_index(KEYS)
        .sort_index()
    )

    assert indexed.index.is_unique

    indexed_tables.append(
        (source_name, indexed)
    )

base_index = indexed_tables[0][1].index

for source_name, indexed in indexed_tables[1:]:

    assert indexed.index.equals(base_index), (
        f"User-window keys differ: {source_name}"
    )


# ------------------------------------------------------------
# 3. MERGE FEATURES WITHOUT DUPLICATING SHARED COLUMNS
# ------------------------------------------------------------

combined_parts = []
registered_columns = {}

for source_name, indexed in indexed_tables:

    new_columns = []

    for col in indexed.columns:

        if col in registered_columns:

            # Shared features must contain the same
            # underlying values, not merely the same name.
            pd.testing.assert_series_equal(
                registered_columns[col],
                indexed[col],
                check_names=False,
                check_dtype=False,
                obj=f"Shared column '{col}' in {source_name}"
            )

        else:
            registered_columns[col] = indexed[col]
            new_columns.append(col)

    if new_columns:
        combined_parts.append(indexed[new_columns])

window_audit_indexed = pd.concat(
    combined_parts,
    axis=1
)

assert window_audit_indexed.index.equals(base_index)
assert not window_audit_indexed.columns.duplicated().any()


# ------------------------------------------------------------
# 4. ADD LONG-ROTATION CAPACITY
# ------------------------------------------------------------

long_frame = standardize_keys(
    long_rotation_capacity
)

long_indexed = (
    long_frame.set_index(KEYS)
    .sort_index()
)

assert long_indexed.index.is_unique

assert long_indexed.index.isin(base_index).all(), (
    "Long-rotation keys missing from main window audit"
)

# Reconcile actual window counts after aligning indexes.
left_windows = long_indexed["eligible_windows"].sort_index()

right_windows = window_audit_indexed.loc[
    long_indexed.index,
    "eligible_windows"
].sort_index()

pd.testing.assert_series_equal(
    left_windows,
    right_windows,
    check_names=False,
    check_dtype=False,
    obj="Long-rotation eligible-window reconciliation"
)

# Keep only genuinely new long-rotation fields.
long_extra = long_indexed.drop(
    columns=["eligible_windows"]
)

collisions = (
    set(long_extra.columns)
    & set(window_audit_indexed.columns)
)

assert not collisions, (
    f"Unexpected long-rotation column collisions: {collisions}"
)

window_audit_indexed = window_audit_indexed.join(
    long_extra,
    how="left",
    validate="one_to_one"
)

window_audit_indexed["is_multichannel_candidate"] = (
    window_audit_indexed.index.isin(
        long_indexed.index
    )
)

user_window_length_audit = (
    window_audit_indexed
    .reset_index()
    .sort_values(KEYS)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. CONSOLIDATE CADENCE + RECONSTRUCTED SEQUENCES
# ------------------------------------------------------------

cadence_frame = standardize_keys(
    rotation_cadence_capacity,
    include_gap=True
)

structure_frame = standardize_keys(
    rotation_structure_detail,
    include_gap=True
)

assert not cadence_frame.duplicated(
    CADENCE_KEYS
).any()

assert not structure_frame.duplicated(
    STRUCTURE_KEYS
).any()

rotation_feasibility_audit = (
    structure_frame.merge(
        cadence_frame,
        on=CADENCE_KEYS,
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

assert rotation_feasibility_audit[
    "_merge"
].eq("both").all(), (
    "Some reconstructed sequences lack cadence records"
)

rotation_feasibility_audit = (
    rotation_feasibility_audit
    .drop(columns="_merge")
    .sort_values(STRUCTURE_KEYS)
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. VALIDATE MAIN USER-WINDOW TABLE
# ------------------------------------------------------------

assert len(user_window_length_audit) == 387162

assert user_window_length_audit[
    "user"
].nunique() == 3737

assert not user_window_length_audit.duplicated(
    KEYS
).any()

assert not user_window_length_audit.columns.duplicated().any()

assert (
    user_window_length_audit["eligible_windows"].sum()
    == 20436421
)

assert (
    user_window_length_audit[
        "calendar_contiguous_windows"
    ]
    + user_window_length_audit[
        "cross_gap_windows"
    ]
    == user_window_length_audit[
        "eligible_windows"
    ]
).all()

# All original audit features must be complete.
original_cols = KEYS + list(
    registered_columns.keys()
)

assert not user_window_length_audit[
    original_cols
].isna().any().any()

# Long-rotation records apply only to multi-channel
# candidates, and must not be converted to zero
# for users who were not analyzed.
candidate_mask = user_window_length_audit[
    "is_multichannel_candidate"
]

assert int(candidate_mask.sum()) == len(long_frame)

long_cols = list(long_extra.columns)

assert not user_window_length_audit.loc[
    candidate_mask,
    long_cols
].isna().any().any()

assert user_window_length_audit.loc[
    ~candidate_mask,
    long_cols
].isna().all().all()

assert user_window_length_audit.loc[
    candidate_mask,
    "user"
].nunique() == 995


# ------------------------------------------------------------
# 7. VALIDATE ROTATION FEASIBILITY TABLE
# ------------------------------------------------------------

assert len(cadence_frame) == 17799
assert len(rotation_feasibility_audit) == 22296

assert not rotation_feasibility_audit.duplicated(
    STRUCTURE_KEYS
).any()

assert not rotation_feasibility_audit.columns.duplicated().any()

assert (
    rotation_feasibility_audit["rotation_steps"]
    <= rotation_feasibility_audit[
        "maximum_rotation_steps"
    ]
).all()

assert (
    rotation_feasibility_audit[
        "channel_switches"
    ]
    == rotation_feasibility_audit[
        "rotation_steps"
    ] - 1
).all()

assert (
    rotation_feasibility_audit[
        "total_unselected_days"
    ]
    == rotation_feasibility_audit[
        "window_workdays"
    ]
    - rotation_feasibility_audit[
        "rotation_steps"
    ]
).all()

assert (
    rotation_feasibility_audit[
        "email_steps"
    ]
    + rotation_feasibility_audit[
        "usb_steps"
    ]
    + rotation_feasibility_audit[
        "http_steps"
    ]
    == rotation_feasibility_audit[
        "rotation_steps"
    ]
).all()

assert set(
    rotation_feasibility_audit[
        "minimum_interstep_offdays"
    ].unique()
) == {0, 1, 2}

assert set(
    rotation_feasibility_audit[
        "window_workdays"
    ].unique()
) == {5, 10, 20, 30, 40, 50}

# Every cadence record must have exactly one
# reconstructed maximum-rotation sequence.
main_paths = rotation_feasibility_audit.loc[
    rotation_feasibility_audit[
        "sequence_variant"
    ] == "maximum_rotation"
]

assert len(main_paths) == len(cadence_frame)

assert not main_paths.duplicated(
    CADENCE_KEYS
).any()

# Three-channel alternatives must genuinely use
# all three channels.
three_paths = rotation_feasibility_audit.loc[
    rotation_feasibility_audit[
        "sequence_variant"
    ] == "three_channel_alternative"
]

assert three_paths[
    "used_channel_count"
].eq(3).all()

assert (
    three_paths["rotation_steps"]
    <= three_paths["maximum_3channel_steps"]
).all()

# Gap 0 reconstruction must attain the global maximum.
gap_zero_main = main_paths.loc[
    main_paths[
        "minimum_interstep_offdays"
    ] == 0
]

assert (
    gap_zero_main["rotation_steps"]
    == gap_zero_main["maximum_rotation_steps"]
).all()


# ------------------------------------------------------------
# 8. WRITE TEMPORARY OUTPUTS AND VERIFY FIRST
# ------------------------------------------------------------

REPORT_DIR_02.mkdir(
    parents=True,
    exist_ok=True
)

outputs = [
    (
        REPORT_DIR_02 / "user_window_length_audit.parquet",
        user_window_length_audit
    ),
    (
        REPORT_DIR_02 / "rotation_feasibility_audit.parquet",
        rotation_feasibility_audit
    )
]

temporary_outputs = []

for final_path, frame in outputs:

    temporary_path = final_path.with_suffix(
        ".parquet.tmp"
    )

    frame.to_parquet(
        temporary_path,
        index=False
    )

    metadata = pq.ParquetFile(
        temporary_path
    )

    assert metadata.metadata.num_rows == len(frame)

    assert (
        metadata.schema_arrow.names
        == list(frame.columns)
    )

    temporary_outputs.append(
        (temporary_path, final_path)
    )

# Finalize only after both temporary outputs pass.
for temporary_path, final_path in temporary_outputs:
    temporary_path.replace(final_path)


# ------------------------------------------------------------
# 9. FINAL CONSOLIDATION REPORT
# ------------------------------------------------------------

print("=" * 78)
print("NOTEBOOK 02 — VALIDATED CONSOLIDATION & EXPORT")
print("=" * 78)

print("\nKEY STANDARDIZATION")
print("-" * 78)
print("window_workdays dtype :", "int64")
print("Key alignment         : PASS")
print("Shared column equality: PASS")

print("\nUSER-WINDOW-LENGTH AUDIT")
print("-" * 78)
print("Rows   :", f"{len(user_window_length_audit):,}")
print("Columns:", len(user_window_length_audit.columns))
print(
    "Users  :",
    f"{user_window_length_audit['user'].nunique():,}"
)
print(
    "Multi-channel candidate records:",
    f"{int(candidate_mask.sum()):,}"
)

print("\nROTATION FEASIBILITY AUDIT")
print("-" * 78)
print("Rows   :", f"{len(rotation_feasibility_audit):,}")
print("Columns:", len(rotation_feasibility_audit.columns))
print(
    "Users  :",
    f"{rotation_feasibility_audit['user'].nunique():,}"
)

print("\nSAVED FILES")
print("-" * 78)

for _, final_path in temporary_outputs:
    print(
        final_path.name,
        f"({final_path.stat().st_size / 1024**2:.2f} MB)"
    )

print("\nEXPORT VALIDATION: PASS")
print("Original Baseline and Future datasets unchanged.")
print("No attack injection performed.")


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\hp\\Documents\\FYP_Data_Exfiltration_Organized\\FYP_Data_Exfiltration\\New_Attack_Simulation\\reports\\02_Temporal_Window_Audit\\rotation_feasibility_audit.parquet.tmp' -> 'C:\\Users\\hp\\Documents\\FYP_Data_Exfiltration_Organized\\FYP_Data_Exfiltration\\New_Attack_Simulation\\reports\\02_Temporal_Window_Audit\\rotation_feasibility_audit.parquet'

In [20]:

# ============================================================
# 6.2 RECOVER AND VERIFY INTERRUPTED PARQUET EXPORT
# ============================================================

import gc
from pathlib import Path
import pyarrow.parquet as pq

# ------------------------------------------------------------
# 1. RELEASE OPEN PARQUET HANDLES
# ------------------------------------------------------------

if "metadata" in globals():
    try:
        metadata.close()
    except (AttributeError, ValueError):
        pass

    del metadata

gc.collect()

print("=" * 75)
print("NOTEBOOK 02 — EXPORT RECOVERY")
print("=" * 75)

# ------------------------------------------------------------
# 2. DEFINE EXPECTED OUTPUTS
# ------------------------------------------------------------

expected_outputs = [
    (
        REPORT_DIR_02 / "user_window_length_audit.parquet",
        user_window_length_audit
    ),
    (
        REPORT_DIR_02 / "rotation_feasibility_audit.parquet",
        rotation_feasibility_audit
    )
]

# ------------------------------------------------------------
# 3. VERIFY FILES BEFORE FINALIZING
# ------------------------------------------------------------

for final_path, expected_frame in expected_outputs:

    temporary_path = final_path.with_suffix(
        ".parquet.tmp"
    )

    # Prefer the temporary file when available.
    source_path = (
        temporary_path
        if temporary_path.exists()
        else final_path
    )

    assert source_path.exists(), (
        f"Missing output: {final_path.name}"
    )

    # These functions read and release file handles.
    file_metadata = pq.read_metadata(source_path)
    file_schema = pq.read_schema(source_path)

    assert file_metadata.num_rows == len(
        expected_frame
    ), f"Row count mismatch: {source_path.name}"

    assert file_schema.names == list(
        expected_frame.columns
    ), f"Column mismatch: {source_path.name}"

    print(f"\nVERIFIED: {source_path.name}")
    print(f"Rows   : {file_metadata.num_rows:,}")
    print(f"Columns: {len(file_schema.names)}")

# ------------------------------------------------------------
# 4. FINALIZE PENDING TEMPORARY FILES
# ------------------------------------------------------------

gc.collect()

for final_path, _ in expected_outputs:

    temporary_path = final_path.with_suffix(
        ".parquet.tmp"
    )

    if temporary_path.exists():

        temporary_path.replace(final_path)

        print(f"\nFINALIZED: {final_path.name}")

    else:
        print(f"\nALREADY FINALIZED: {final_path.name}")

# ------------------------------------------------------------
# 5. FINAL ON-DISK VERIFICATION
# ------------------------------------------------------------

for final_path, expected_frame in expected_outputs:

    assert final_path.is_file()

    file_metadata = pq.read_metadata(final_path)
    file_schema = pq.read_schema(final_path)

    assert file_metadata.num_rows == len(
        expected_frame
    )

    assert file_schema.names == list(
        expected_frame.columns
    )

    assert not final_path.with_suffix(
        ".parquet.tmp"
    ).exists()

    print(
        f"PASS | {final_path.name} | "
        f"{file_metadata.num_rows:,} rows | "
        f"{len(file_schema.names)} columns"
    )

print("\n" + "=" * 75)
print("EXPORT RECOVERY: PASS")
print("Both Parquet files verified and finalized.")
print("Original source datasets unchanged.")
print("=" * 75)


NOTEBOOK 02 — EXPORT RECOVERY

VERIFIED: user_window_length_audit.parquet
Rows   : 387,162
Columns: 116

VERIFIED: rotation_feasibility_audit.parquet.tmp
Rows   : 22,296
Columns: 52

ALREADY FINALIZED: user_window_length_audit.parquet

FINALIZED: rotation_feasibility_audit.parquet
PASS | user_window_length_audit.parquet | 387,162 rows | 116 columns
PASS | rotation_feasibility_audit.parquet | 22,296 rows | 52 columns

EXPORT RECOVERY: PASS
Both Parquet files verified and finalized.
Original source datasets unchanged.



## 7. Data Dictionary Preparation

### 7.1 Saved Output Schema Inventory

**Objective:** Inspect the actual saved Parquet schemas before creating the final Data Dictionary.

For every column, inspect:
- Column name
- Data type
- Null count
- Source output file

The final dictionary will document each column's meaning, calculation, source, unit, and interpretation, including whether missing values mean "not applicable."

**Validation:** All 168 column occurrences across both files must be documented.

No source modifications or new files are created in this step.


In [21]:

# ============================================================
# 7.1 SAVED OUTPUT SCHEMA INVENTORY
# ============================================================

import pandas as pd

files_to_inspect = {
    "user_window_length_audit.parquet":
        REPORT_DIR_02 / "user_window_length_audit.parquet",

    "rotation_feasibility_audit.parquet":
        REPORT_DIR_02 / "rotation_feasibility_audit.parquet"
}

schema_parts = []

for filename, path in files_to_inspect.items():

    assert path.is_file(), f"Missing file: {path}"

    df = pd.read_parquet(path)

    schema = pd.DataFrame({
        "file": filename,
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "null_count": df.isna().sum().to_numpy()
    })

    assert schema["column"].is_unique

    schema_parts.append(schema)

schema_inventory = pd.concat(
    schema_parts,
    ignore_index=True
)

assert len(schema_inventory) == 168

print("=" * 80)
print("NOTEBOOK 02 — SAVED SCHEMA INVENTORY")
print("=" * 80)

for filename, part in schema_inventory.groupby(
    "file", sort=False
):
    print(f"\nFILE: {filename}")
    print(f"COLUMNS: {len(part)}")
    print("-" * 80)

    for i, row in enumerate(part.itertuples(), start=1):
        print(
            f"{i:03d} | {row.column} | "
            f"{row.dtype} | nulls={row.null_count}"
        )

print("\n" + "=" * 80)
print("TOTAL DOCUMENTATION TARGET:", len(schema_inventory))
print("SCHEMA INVENTORY: PASS")
print("No files written.")


NOTEBOOK 02 — SAVED SCHEMA INVENTORY

FILE: user_window_length_audit.parquet
COLUMNS: 116
--------------------------------------------------------------------------------
001 | user | str | nulls=0
002 | window_workdays | int64 | nulls=0
003 | eligible_windows | int64 | nulls=0
004 | calendar_contiguous_windows | int64 | nulls=0
005 | cross_gap_windows | int64 | nulls=0
006 | has_calendar_contiguous_option | bool | nulls=0
007 | windows_NONE | int64 | nulls=0
008 | windows_E | int64 | nulls=0
009 | windows_U | int64 | nulls=0
010 | windows_EU | int64 | nulls=0
011 | windows_H | int64 | nulls=0
012 | windows_EH | int64 | nulls=0
013 | windows_UH | int64 | nulls=0
014 | windows_EUH | int64 | nulls=0
015 | windows_one_observed_channel | int64 | nulls=0
016 | windows_two_observed_channels | int64 | nulls=0
017 | windows_three_observed_channels | int64 | nulls=0
018 | windows_2plus_observed_channels | int64 | nulls=0
019 | windows_any_observed_channel | int64 | nulls=0
020 | observed_2plus_


## 7.2 Complete Data Dictionary Generation

**Objective:** Create a validated Data Dictionary for the two finalized Notebook 02 Parquet outputs.

**Documentation Scope:**
- 116 columns from `user_window_length_audit.parquet`
- 52 columns from `rotation_feasibility_audit.parquet`
- 168 column definitions in total

**Each Definition Includes:**
- Column name and data type
- Row granularity and metric category
- Detailed meaning and calculation
- Source table or analysis
- Unit and valid values
- Null count, null percentage, and missing-value interpretation
- Observed minimum, maximum, and example value
- Interpretation notes and documentation status

**Validation Requirements:**
- Every actual column must have exactly one definition per file.
- No undocumented or unexpected columns are allowed.
- Long-rotation null values must match the noncandidate population.
- The saved dictionary must pass a read-back verification.

**Output:**
`reports/02_Temporal_Window_Audit/notebook02_data_dictionary.csv`

Original datasets and validated Parquet outputs remain unchanged.


In [22]:

# ============================================================
# 7.2 BUILD & VALIDATE COMPLETE DATA DICTIONARY
# ============================================================

from pathlib import Path

DICTIONARY_SCRIPT = (
    PROJECT_ROOT
    / "MUBSIR_NB02_Data_Dictionary_Builder.py"
)

assert DICTIONARY_SCRIPT.is_file(), (
    f"Data Dictionary script not found: {DICTIONARY_SCRIPT}"
)

exec(
    compile(
        DICTIONARY_SCRIPT.read_text(encoding="utf-8"),
        str(DICTIONARY_SCRIPT),
        "exec"
    ),
    globals()
)


NOTEBOOK 02 — DATA DICTIONARY VALIDATION
PASS | user_window_length_audit.parquet: 116/116 columns documented
PASS | rotation_feasibility_audit.parquet: 52/52 columns documented
PASS | Complete definitions: 168/168
PASS | Exact names, dtypes and null counts obtained from saved Parquet files
PASS | Long-rotation missing values reconciled with noncandidate users
PASS | Saved CSV round-trip verified
SAVED: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\02_Temporal_Window_Audit\notebook02_data_dictionary.csv
Data Dictionary: PASS — 168/168 documented
Baseline and source Future remain untouched
